# Dataset and split audits, reviewer checks, original inference provenance

These are original, unmodified Python source cells from the exported Kaggle notebook, organized by research phase. Execution outputs and counts have been cleared. Cells are in original index order within this phase. The Kaggle project workspace and some earlier phases are required; this notebook is not guaranteed to execute by itself from a clean environment. The original index for every cell is stored in cell metadata.

The historical record includes exploratory retries and some superseded checks. Do not rerun an early output-producing cell over frozen results without backing up the project first.

In [ ]:

import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

P = Path("/kaggle/working/acoustic_linguistic_research")

DATA = P / "data/modeling/modeling_dataset_0001-0400.csv"
SPLIT = P / "data/splits/speaker_groupkfold5_frozen.csv"
DESIGN = P / "configs/fluency_feature_design_v1.json"
CV = P / "configs/speaker_disjoint_cv_v1.json"

OUT = P / "results/step07_methodology"
REPORT = OUT / "step07_01_dataset_integrity.json"
SPEAKERS = OUT / "step07_01_speaker_summary.csv"
FOLDS = OUT / "step07_01_fold_summary.csv"

EXPECTED = {
    DATA: "7d20428a14d9e00b88f1a20b36b7147bf5b98234045a093172fef76b68276746",
    SPLIT: "5f87cdc1c2f69922bcb542ee9516f5f89168bf68402fbd2449b28fcf61a375e7",
    DESIGN: "626b0050f58f35fafa4f2efcd1888fda2a195454e544c4cb094cd50e796a1b83",
    CV: "302c7ea0086a54d644a4ebc0b8a5d96d63575a3896b5a29d76dba7e26dacd472"
}

def sha(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

for path, expected in EXPECTED.items():
    assert path.is_file(), f"Missing: {path}"
    assert sha(path) == expected, f"SHA mismatch: {path}"

data = pd.read_csv(
    DATA,
    dtype={"audio_id": "string", "speaker": "string"}
)

split = pd.read_csv(
    SPLIT,
    dtype={"audio_id": "string", "speaker": "string"}
)

design = json.loads(DESIGN.read_text(encoding="utf-8"))
cv = json.loads(CV.read_text(encoding="utf-8"))

IDS = ["pilot_index", "audio_id", "speaker"]
KEYS = IDS + ["hypothesis_id"]

assert len(data) == 2000
assert len(split) == 400
assert data["audio_id"].nunique() == 400
assert split["audio_id"].nunique() == 400
assert data["speaker"].nunique() == 20
assert split["speaker"].nunique() == 20

assert not data.duplicated(KEYS).any()
assert not split.duplicated(IDS).any()

assert data.groupby("audio_id")["hypothesis_id"].nunique().eq(5).all()
assert data.groupby("audio_id")["fluency"].nunique().eq(1).all()
assert data.groupby("audio_id")["speaker"].nunique().eq(1).all()

assert set(data["audio_id"]) == set(split["audio_id"])

merged = data.merge(
    split[IDS + ["fluency", "validation_fold"]],
    on=IDS,
    how="inner",
    suffixes=("_data", "_split"),
    validate="many_to_one"
)

assert len(merged) == 2000
assert np.allclose(
    merged["fluency_data"],
    merged["fluency_split"]
)

assert merged["validation_fold"].nunique() == 5

assert merged.groupby("speaker")[
    "validation_fold"
].nunique().eq(1).all()

assert merged.groupby("audio_id")[
    "validation_fold"
].nunique().eq(1).all()

assert set(merged["validation_fold"]) == {1, 2, 3, 4, 5}

# One independent observation per original audio.
audio = split.copy()

assert audio.groupby("speaker")["audio_id"].nunique().eq(20).all()

speaker_summary = (
    audio.groupby(["speaker", "validation_fold"])
    .agg(
        audios=("audio_id", "nunique"),
        fluency_mean=("fluency", "mean"),
        fluency_sd=("fluency", "std"),
        fluency_min=("fluency", "min"),
        fluency_max=("fluency", "max")
    )
    .reset_index()
    .sort_values(["validation_fold", "speaker"])
)

fold_summary = (
    audio.groupby("validation_fold")
    .agg(
        speakers=("speaker", "nunique"),
        audios=("audio_id", "nunique"),
        fluency_mean=("fluency", "mean"),
        fluency_sd=("fluency", "std"),
        fluency_min=("fluency", "min"),
        fluency_max=("fluency", "max")
    )
    .reset_index()
)

assert len(speaker_summary) == 20
assert len(fold_summary) == 5
assert fold_summary["audios"].eq(80).all()
assert fold_summary["speakers"].eq(4).all()

# Verify that the five folds contain disjoint speakers.
fold_speakers = {
    int(fold): set(group["speaker"])
    for fold, group in audio.groupby("validation_fold")
}

for fold_a in range(1, 6):
    for fold_b in range(fold_a + 1, 6):
        assert fold_speakers[fold_a].isdisjoint(
            fold_speakers[fold_b]
        )

LING = design["linguistic_feature_names"]
PROMPT = design["prompt_feature_names"]

ACOUSTIC = [
    c for c in data.columns
    if c not in KEYS + ["fluency"] + LING + PROMPT
]

assert len(ACOUSTIC) == 14
assert len(LING) == 7
assert len(PROMPT) == 2

missing = {
    col: {
        "count": int(data[col].isna().sum()),
        "percent": float(data[col].isna().mean() * 100)
    }
    for col in ACOUSTIC + LING + PROMPT
    if data[col].isna().any()
}

feature_groups = {
    "acoustic": ACOUSTIC,
    "linguistic": LING,
    "prompt": PROMPT
}

REPORT_DATA = {
    "status": "PASS",
    "step": "7.1",
    "analysis": "dataset_split_and_feature_integrity",
    "dataset": "mispeech/speechocean762",
    "dataset_revision": "06385584fad212b26134c656fdd3ccf9f093f33e",
    "dataset_partition_analyzed": "train",
    "official_test_access_checked_by_this_cell": False,
    "audio_count": int(audio["audio_id"].nunique()),
    "speaker_count": int(audio["speaker"].nunique()),
    "hypothesis_rows": int(len(data)),
    "hypotheses_per_audio": 5,
    "fold_count": 5,
    "validation_speakers_per_fold": 4,
    "validation_audios_per_fold": 80,
    "human_fluency_min": float(audio["fluency"].min()),
    "human_fluency_max": float(audio["fluency"].max()),
    "human_fluency_mean": float(audio["fluency"].mean()),
    "human_fluency_sd": float(audio["fluency"].std()),
    "feature_groups": feature_groups,
    "missing_feature_values": missing,
    "speaker_disjoint": True,
    "audio_disjoint": True,
    "target_consistency": True,
    "all_original_audios_retained": True,
    "models_retrained": False,
    "source_sha256": {
        path.name: sha(path)
        for path in EXPECTED
    },
    "methodology_notes": [
        "Five ASR hypotheses share one human fluency label.",
        "The effective independent sampling units are speakers.",
        "Fold mean labels may differ by speaker composition.",
        "Dataset revision is documented from frozen experiment records.",
        "This cell does not independently verify initial sample randomization.",
        "This cell does not independently prove the official test was never opened.",
        "Prompt text is not verified ground-truth transcription."
    ]
}

OUT.mkdir(parents=True, exist_ok=True)

def safe_csv(frame, path):
    temp = path.with_suffix(".tmp")
    frame.to_csv(
        temp,
        index=False,
        encoding="utf-8",
        float_format="%.12g"
    )

    if path.exists():
        assert sha(path) == sha(temp), (
            f"Existing report differs: {path}"
        )
        temp.unlink()
    else:
        temp.replace(path)

safe_csv(speaker_summary, SPEAKERS)
safe_csv(fold_summary, FOLDS)

REPORT_DATA["speaker_summary_sha256"] = sha(SPEAKERS)
REPORT_DATA["fold_summary_sha256"] = sha(FOLDS)

if REPORT.exists():
    assert json.loads(
        REPORT.read_text(encoding="utf-8")
    ) == REPORT_DATA, "Existing JSON differs"
else:
    temp = REPORT.with_suffix(".tmp")
    temp.write_text(
        json.dumps(
            REPORT_DATA,
            indent=2,
            ensure_ascii=False
        ),
        encoding="utf-8"
    )
    temp.replace(REPORT)

print("=== STEP 7.1 DATASET INTEGRITY AUDIT ===")
print("Status: PASS")
print("Original audios:", len(audio))
print("Speakers:", audio["speaker"].nunique())
print("ASR hypotheses:", len(data))
print("Speaker-disjoint folds: PASS")
print("Target consistency: PASS")

print("\n=== FLUENCY DISTRIBUTION ===")
print(
    "Min:", REPORT_DATA["human_fluency_min"],
    "Max:", REPORT_DATA["human_fluency_max"],
    "Mean:", round(REPORT_DATA["human_fluency_mean"], 4),
    "SD:", round(REPORT_DATA["human_fluency_sd"], 4)
)

print("\n=== FOLD SUMMARY ===")
print(fold_summary.round(4).to_string(index=False))

print("\n=== SPEAKER SUMMARY ===")
print(speaker_summary.round(4).to_string(index=False))

print("\n=== FEATURE MISSINGNESS ===")
print(json.dumps(missing, indent=2))

print("\nReport:", REPORT)
print("SHA256:", sha(REPORT))
print("Dataset and split unchanged: YES")
print("Next: STEP 7.2 leakage audit")


In [ ]:

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

P = Path("/kaggle/working/acoustic_linguistic_research")
DATA = P / "data/modeling/modeling_dataset_0001-0400.csv"
SPLIT = P / "data/splits/speaker_groupkfold5_frozen.csv"
DESIGN = P / "configs/fluency_feature_design_v1.json"
C2 = P / "results/step05/c2_prompt_aware_fusion_ridge_oof_predictions.csv"
OUT = P / "results/step07_methodology"
REPORT = OUT / "step07_02_leakage_audit.json"
FOLD_CSV = OUT / "step07_02_fold_preprocessing_checks.csv"

HASHES = {
    DATA: "7d20428a14d9e00b88f1a20b36b7147bf5b98234045a093172fef76b68276746",
    SPLIT: "5f87cdc1c2f69922bcb542ee9516f5f89168bf68402fbd2449b28fcf61a375e7",
    DESIGN: "626b0050f58f35fafa4f2efcd1888fda2a195454e544c4cb094cd50e796a1b83",
    C2: "2592e8d28c656db43280d6eea30d578cd488d75f0382e3ed083abf5e2ab86df9"
}

def sha(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

for path, expected in HASHES.items():
    assert path.is_file()
    assert sha(path) == expected, f"SHA mismatch: {path}"

def read(path):
    return pd.read_csv(
        path,
        dtype={"audio_id": "string", "speaker": "string"}
    )

data = read(DATA)
split = read(SPLIT)
saved = read(C2)
design = json.loads(DESIGN.read_text(encoding="utf-8"))

IDS = ["pilot_index", "audio_id", "speaker"]
KEYS = IDS + ["hypothesis_id"]
FULL = KEYS + ["fluency", "validation_fold"]

LING = list(design["linguistic_feature_names"])
PROMPT = list(design["prompt_feature_names"])

ACOUSTIC = [
    c for c in data.columns
    if c not in KEYS + ["fluency"] + LING + PROMPT
]

FEATURES = ACOUSTIC + LING + PROMPT

assert len(ACOUSTIC) == 14
assert len(LING) == 7
assert len(PROMPT) == 2
assert len(FEATURES) == 23
assert len(set(FEATURES)) == 23

FORBIDDEN = {
    "fluency", "audio_id", "speaker", "pilot_index",
    "hypothesis_id", "validation_fold",
    "c2_prediction", "human_score", "human_fluency"
}

assert FORBIDDEN.isdisjoint(FEATURES)

df = data.merge(
    split[IDS + ["fluency", "validation_fold"]],
    on=IDS + ["fluency"],
    validate="many_to_one",
    how="inner"
)

assert len(df) == 2000
assert df["audio_id"].nunique() == 400
assert df["speaker"].nunique() == 20

assert not df.duplicated(KEYS).any()
assert df.groupby("audio_id")["hypothesis_id"].nunique().eq(5).all()
assert df.groupby("audio_id")["validation_fold"].nunique().eq(1).all()
assert df.groupby("speaker")["validation_fold"].nunique().eq(1).all()

assert len(saved) == 2000
assert not saved.duplicated(KEYS).any()

fold_results = []
predictions = []

for fold in range(1, 6):
    train = df.loc[df["validation_fold"] != fold].copy()
    val = df.loc[df["validation_fold"] == fold].copy()

    assert len(train) == 1600
    assert len(val) == 400

    train_speakers = set(train["speaker"])
    val_speakers = set(val["speaker"])

    train_audios = set(train["audio_id"])
    val_audios = set(val["audio_id"])

    assert train_speakers.isdisjoint(val_speakers)
    assert train_audios.isdisjoint(val_audios)

    assert len(train_speakers) == 16
    assert len(val_speakers) == 4
    assert len(train_audios) == 320
    assert len(val_audios) == 80

    X_train = train[FEATURES]
    X_val = val[FEATURES]

    assert all(
        pd.api.types.is_numeric_dtype(X_train[col])
        for col in FEATURES
    )

    # Independently compute train-only median statistics.
    train_medians = X_train.median(axis=0).to_numpy(dtype=float)

    model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=1.0))
    ])

    model.fit(X_train, train["fluency"])

    imputer = model.named_steps["imputer"]
    scaler = model.named_steps["scaler"]

    assert np.allclose(
        imputer.statistics_,
        train_medians,
        rtol=1e-12,
        atol=1e-12
    )

    train_imputed = imputer.transform(X_train)

    independent_mean = train_imputed.mean(axis=0)
    independent_var = train_imputed.var(axis=0)

    assert np.allclose(
        scaler.mean_,
        independent_mean,
        rtol=1e-12,
        atol=1e-12
    )
    assert np.allclose(
        scaler.var_,
        independent_var,
        rtol=1e-12,
        atol=1e-12
    )

    # Compare against a different hypothetical preprocessing:
    # fitting on all data. This is diagnostic only;
    # those statistics are NEVER used for prediction.
    train_missing = int(X_train.isna().sum().sum())
    val_missing = int(X_val.isna().sum().sum())

    pred = model.predict(X_val)
    assert np.isfinite(pred).all()

    part = val[FULL].copy()
    part["reconstructed_prediction"] = pred
    predictions.append(part)

    fold_results.append({
        "fold": fold,
        "train_speakers": len(train_speakers),
        "validation_speakers": len(val_speakers),
        "train_audios": len(train_audios),
        "validation_audios": len(val_audios),
        "train_hypotheses": len(train),
        "validation_hypotheses": len(val),
        "speaker_overlap": len(train_speakers & val_speakers),
        "audio_overlap": len(train_audios & val_audios),
        "training_missing_cells": train_missing,
        "validation_missing_cells": val_missing,
        "train_only_median_verified": True,
        "train_only_scaler_verified": True
    })

reconstructed = pd.concat(predictions, ignore_index=True)

comparison = reconstructed.merge(
    saved[FULL + ["c2_prediction"]],
    on=FULL,
    how="inner",
    validate="one_to_one"
)

assert len(comparison) == 2000

delta = np.abs(
    comparison["reconstructed_prediction"].to_numpy()
    - comparison["c2_prediction"].to_numpy()
)

max_delta = float(delta.max())
assert max_delta < 1e-7, (
    f"Reconstruction mismatch: {max_delta}"
)

fold_table = pd.DataFrame(fold_results)

assert len(fold_table) == 5
assert fold_table["speaker_overlap"].eq(0).all()
assert fold_table["audio_overlap"].eq(0).all()

OUT.mkdir(parents=True, exist_ok=True)

def safe_csv(frame, path):
    temp = path.with_suffix(".tmp")
    frame.to_csv(temp, index=False, encoding="utf-8")

    if path.exists():
        assert sha(path) == sha(temp), (
            f"Existing CSV differs: {path}"
        )
        temp.unlink()
    else:
        temp.replace(path)

safe_csv(fold_table, FOLD_CSV)

report = {
    "status": "PASS",
    "step": "7.2",
    "audit": "speaker_split_feature_and_preprocessing_leakage",
    "speaker_disjoint": True,
    "audio_disjoint": True,
    "hypotheses_grouped_by_audio": True,
    "forbidden_predictor_columns_excluded": True,
    "imputation_fit_on_training_fold": True,
    "scaler_fit_on_training_fold": True,
    "human_fluency_is_training_target_only": True,
    "validation_labels_not_used_for_preprocessing": True,
    "ridge_alpha": 1.0,
    "oof_predictions_reconstructed": 2000,
    "max_prediction_absolute_difference": max_delta,
    "fold_results_sha256": sha(FOLD_CSV),
    "source_sha256": {
        path.name: sha(path)
        for path in HASHES
    },
    "scope_limitations": [
        "Confirms leakage controls for reconstructed C2 pipeline.",
        "Does not independently audit original ASR training data.",
        "Does not verify possible overlap with Whisper pretraining.",
        "Does not establish provenance of source audio or prompts.",
        "Does not audit original sample selection procedure.",
        "Does not independently prove sealed test access history.",
        "Does not establish whether prompt features are permitted operationally."
    ]
}

if REPORT.exists():
    old = json.loads(REPORT.read_text(encoding="utf-8"))
    assert old == report, "Existing report differs"
else:
    temp = REPORT.with_suffix(".tmp")
    temp.write_text(
        json.dumps(report, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    temp.replace(REPORT)

print("=== STEP 7.2 DATA LEAKAGE AUDIT ===")
print("Status: PASS")
print("Feature count:", len(FEATURES))
print("Forbidden predictors: EXCLUDED")
print("Speaker overlap: ZERO")
print("Audio overlap: ZERO")
print("Train-only imputation: PASS")
print("Train-only scaling: PASS")
print("C2 OOF reconstruction: PASS")
print("Maximum prediction difference:", max_delta)

print("\n=== FOLD-BY-FOLD LEAKAGE CHECKS ===")
print(fold_table.to_string(index=False))

print("\n=== FEATURE GROUPS ===")
print("Acoustic:", len(ACOUSTIC))
print("Linguistic:", len(LING))
print("Prompt-aware:", len(PROMPT))

print("\n=== SCOPE LIMITATIONS ===")
for item in report["scope_limitations"]:
    print("-", item)

print("\nReport:", REPORT)
print("SHA256:", sha(REPORT))
print("Fold CSV:", FOLD_CSV)
print("Next: STEP 7.3 model-comparison fairness")


In [ ]:

import hashlib
import json
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

P = Path("/kaggle/working/acoustic_linguistic_research")
D = P / "data/modeling/modeling_dataset_0001-0400.csv"
S = P / "data/splits/speaker_groupkfold5_frozen.csv"
CONFIG = P / "configs/fluency_feature_design_v1.json"
OUT = P / "results/step07_methodology"
OUT.mkdir(parents=True, exist_ok=True)

SOURCE_HASHES = {
    D: "7d20428a14d9e00b88f1a20b36b7147bf5b98234045a093172fef76b68276746",
    S: "5f87cdc1c2f69922bcb542ee9516f5f89168bf68402fbd2449b28fcf61a375e7",
    CONFIG: "626b0050f58f35fafa4f2efcd1888fda2a195454e544c4cb094cd50e796a1b83"
}

ORIGINALS = {
    "B1": ("b1_transcript_ridge_oof_predictions.csv",
           "b1_prediction",
           "375ac49b0d5b565adf65d04407d6afc1816f64f5ba108afef8d22458383807dc"),
    "B2": ("b2_prompt_aware_ridge_oof_predictions.csv",
           "b2_prediction",
           "ba212868934dbe4d92234f46ccb5f7d7e66a56c781d74aeb24b310b722cfb189"),
    "C1": ("c1_fusion_ridge_oof_predictions.csv",
           "c1_prediction",
           "43aa2b8d73c60e44ee4ee2f996f5b243ae2a09c5b9a78f909b2a56ba7306669e"),
    "C2": ("c2_prompt_aware_fusion_ridge_oof_predictions.csv",
           "c2_prediction",
           "2592e8d28c656db43280d6eea30d578cd488d75f0382e3ed083abf5e2ab86df9")
}

def sha(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

for path, expected in SOURCE_HASHES.items():
    assert path.is_file() and sha(path) == expected

def read(path):
    return pd.read_csv(
        path, dtype={"audio_id": "string", "speaker": "string"}
    )

data = read(D)
split = read(S)
design = json.loads(CONFIG.read_text(encoding="utf-8"))

IDS = ["pilot_index", "audio_id", "speaker"]
KEYS = IDS + ["hypothesis_id"]
FULL = KEYS + ["fluency", "validation_fold"]

LING = list(design["linguistic_feature_names"])
PROMPT = list(design["prompt_feature_names"])
ACOUSTIC = [
    c for c in data.columns
    if c not in KEYS + ["fluency"] + LING + PROMPT
]

FEATURE_MAP = {
    "B1": LING,
    "B2": LING + PROMPT,
    "C1": ACOUSTIC + LING,
    "C2": ACOUSTIC + LING + PROMPT
}

assert [len(FEATURE_MAP[m]) for m in ORIGINALS] == [7, 9, 21, 23]

df = data.merge(
    split[IDS + ["fluency", "validation_fold"]],
    on=IDS + ["fluency"],
    how="inner",
    validate="many_to_one"
)

assert len(df) == 2000
assert df["audio_id"].nunique() == 400
assert df["speaker"].nunique() == 20
assert not df.duplicated(KEYS).any()

saved = {}

for name, (filename, col, expected_hash) in ORIGINALS.items():
    path = P / "results/step05" / filename
    assert sha(path) == expected_hash

    original = read(path)
    assert len(original) == 2000
    assert not original.duplicated(KEYS).any()

    saved[name] = original[FULL + [col]]

rows = []
all_predictions = []
max_reconstruction = {}

for name, features in FEATURE_MAP.items():

    model_parts = []

    for fold in range(1, 6):
        tr = df[df["validation_fold"] != fold]
        va = df[df["validation_fold"] == fold]

        assert len(tr) == 1600 and len(va) == 400
        assert tr["audio_id"].nunique() == 320
        assert va["audio_id"].nunique() == 80
        assert set(tr["speaker"]).isdisjoint(set(va["speaker"]))
        assert set(tr["audio_id"]).isdisjoint(set(va["audio_id"]))

        for alpha in [1.0, 5.0]:

            pipe = Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                ("ridge", Ridge(alpha=alpha))
            ])

            pipe.fit(tr[features], tr["fluency"])
            prediction = pipe.predict(va[features])
            assert np.isfinite(prediction).all()

            part = va[FULL].copy()
            part["model"] = name
            part["alpha"] = alpha
            part["prediction"] = prediction

            model_parts.append(part)

    result = pd.concat(model_parts, ignore_index=True)

    for alpha, sub in result.groupby("alpha"):

        assert len(sub) == 2000
        assert not sub.duplicated(KEYS).any()

        error = (
            sub["prediction"].to_numpy(dtype=float)
            - sub["fluency"].to_numpy(dtype=float)
        )

        ranges = sub.groupby(
            IDS
        )["prediction"].agg(
            lambda x: x.max() - x.min()
        )

        assert len(ranges) == 400

        rows.append({
            "model": name,
            "alpha": float(alpha),
            "features": len(features),
            "OOF_predictions": len(sub),
            "MAE": float(np.abs(error).mean()),
            "RMSE": float(np.sqrt(np.square(error).mean())),
            "mean_prediction_range": float(ranges.mean()),
            "p95_prediction_range": float(ranges.quantile(0.95)),
            "max_prediction_range": float(ranges.max())
        })

        if alpha == 1.0:

            original_col = ORIGINALS[name][1]
            matched = sub.merge(
                saved[name],
                on=FULL,
                how="inner",
                validate="one_to_one"
            )
            assert len(matched) == 2000

            delta = np.abs(
                matched["prediction"].to_numpy()
                - matched[original_col].to_numpy()
            )

            maximum = float(delta.max())
            assert maximum < 1e-7, (
                f"{name} reconstruction mismatch: {maximum}"
            )
            max_reconstruction[name] = maximum

    all_predictions.append(result)

summary = pd.DataFrame(rows)
predictions = pd.concat(all_predictions, ignore_index=True)

assert len(summary) == 8
assert len(predictions) == 16000
assert len(max_reconstruction) == 4

# A is not refitted; its frozen benchmark is retained.
A_MAE = 0.9399
A_RMSE = 1.2066

summary["MAE_minus_A"] = summary["MAE"] - A_MAE
summary["RMSE_minus_A"] = summary["RMSE"] - A_RMSE

paired_rows = []
for alpha in [1.0, 5.0]:
    sub = summary.set_index(["model", "alpha"])

    c2 = sub.loc[("C2", alpha)]
    c1 = sub.loc[("C1", alpha)]

    paired_rows.append({
        "alpha": alpha,
        "C2_minus_C1_MAE": float(c2["MAE"] - c1["MAE"]),
        "C2_minus_C1_RMSE": float(c2["RMSE"] - c1["RMSE"]),
        "C2_minus_A_MAE": float(c2["MAE"] - A_MAE),
        "C2_minus_A_RMSE": float(c2["RMSE"] - A_RMSE),
        "C2_minus_C1_mean_range": float(
            c2["mean_prediction_range"] -
            c1["mean_prediction_range"]
        )
    })

paired = pd.DataFrame(paired_rows)

SUMMARY = OUT / "step07_03_regularization_sensitivity.csv"
PRED = OUT / "step07_03_regularization_oof_predictions.csv"
PAIRED = OUT / "step07_03_regularization_comparisons.csv"
REPORT = OUT / "step07_03_regularization_fairness.json"

def safe_csv(frame, path):
    temp = path.with_suffix(".tmp")
    frame.to_csv(
        temp, index=False, encoding="utf-8",
        float_format="%.12g"
    )
    if path.exists():
        assert sha(path) == sha(temp), (
            f"Existing output differs: {path}"
        )
        temp.unlink()
    else:
        temp.replace(path)

safe_csv(summary, SUMMARY)
safe_csv(predictions, PRED)
safe_csv(paired, PAIRED)

report = {
    "status": "PASS",
    "step": "7.3",
    "analysis": "ridge_regularization_scale_sensitivity",
    "original_alpha": 1.0,
    "relative_penalty_matched_alpha": 5.0,
    "audio_training_rows_for_A": 320,
    "hypothesis_training_rows_for_text_models": 1600,
    "unique_audios": 400,
    "unique_speakers": 20,
    "hypotheses_per_audio": 5,
    "original_OOF_reconstruction_max_difference":
        max_reconstruction,
    "new_model_selection_performed": False,
    "official_test_used": False,
    "primary_original_predictions_unchanged": True,
    "source_hashes": {
        str(path.relative_to(P)): sha(path)
        for path in SOURCE_HASHES
    },
    "output_hashes": {
        path.name: sha(path)
        for path in [SUMMARY, PRED, PAIRED]
    },
    "interpretation": (
        "Alpha 5 is an equal-per-audio objective-normalization "
        "sensitivity check, not a tuned optimum. "
        "For acoustic duplicate rows it matches the alpha 1 "
        "penalty-to-loss ratio of 320 original audio rows. "
        "Transcript features differ across hypotheses, so "
        "this does not make all model assumptions identical."
    )
}

if REPORT.exists():
    assert json.loads(
        REPORT.read_text(encoding="utf-8")
    ) == report, "Existing report differs"
else:
    temp = REPORT.with_suffix(".tmp")
    temp.write_text(
        json.dumps(report, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    temp.replace(REPORT)

print("=== STEP 7.3 REGULARIZATION FAIRNESS AUDIT ===")
print("Status: PASS")
print("Original Ridge alpha:", 1.0)
print("Normalized sensitivity alpha:", 5.0)
print("Original STEP 5 predictions changed: NO")

print("\n=== ORIGINAL PREDICTION RECONSTRUCTION ===")
for name, maximum in max_reconstruction.items():
    print(name, "max absolute difference:", maximum)

print("\n=== ALPHA 1 VS ALPHA 5 ===")
print(
    summary[
        ["model", "alpha", "MAE", "RMSE",
         "mean_prediction_range", "MAE_minus_A", "RMSE_minus_A"]
    ].round(4).to_string(index=False)
)

print("\n=== C2 PAIRED COMPARISONS ===")
print(paired.round(4).to_string(index=False))

print("\nSummary CSV:", SUMMARY)
print("Predictions CSV:", PRED)
print("Report:", REPORT)
print("Report SHA256:", sha(REPORT))
print("Next: interpret fairness sensitivity before Step 7.4")


In [ ]:

import json
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd

P = Path("/kaggle/working/acoustic_linguistic_research")
R = P / "results/step05"
OUT = P / "results/step07_methodology"
CHECK = P / "checkpoints/research_progress.json"

DATA = P / "data/modeling/modeling_dataset_0001-0400.csv"
PROMPT = P / "features/linguistic/prompt_asr_disagreement_0001-0400.csv"
DESIGN = P / "configs/fluency_feature_design_v1.json"

MODELS = {
    "B1": ("b1_transcript_ridge_oof_predictions.csv",
           "b1_prediction"),
    "B2": ("b2_prompt_aware_ridge_oof_predictions.csv",
           "b2_prediction"),
    "C1": ("c1_fusion_ridge_oof_predictions.csv",
           "c1_prediction"),
    "C2": ("c2_prompt_aware_fusion_ridge_oof_predictions.csv",
           "c2_prediction")
}

HASHES = {
    DATA: "7d20428a14d9e00b88f1a20b36b7147bf5b98234045a093172fef76b68276746",
    PROMPT: "8b52805a91144c5ee0a379aeb2954595f55ec085203ac38861845c5fbe33c25c",
    DESIGN: "626b0050f58f35fafa4f2efcd1888fda2a195454e544c4cb094cd50e796a1b83"
}

MODEL_HASHES = {
    "B1": "375ac49b0d5b565adf65d04407d6afc1816f64f5ba108afef8d22458383807dc",
    "B2": "ba212868934dbe4d92234f46ccb5f7d7e66a56c781d74aeb24b310b722cfb189",
    "C1": "43aa2b8d73c60e44ee4ee2f996f5b243ae2a09c5b9a78f909b2a56ba7306669e",
    "C2": "2592e8d28c656db43280d6eea30d578cd488d75f0382e3ed083abf5e2ab86df9"
}

def sha(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def read(path):
    return pd.read_csv(
        path,
        dtype={"audio_id": "string", "speaker": "string"}
    )

for path, expected in HASHES.items():
    assert path.is_file() and sha(path) == expected

design = json.loads(DESIGN.read_text(encoding="utf-8"))
features = list(design["prompt_feature_names"])

assert len(features) == 2

data = read(DATA)
prompt = read(PROMPT)

KEYS = ["pilot_index", "audio_id", "speaker", "hypothesis_id"]
IDS = ["pilot_index", "audio_id", "speaker"]
FULL = KEYS + ["fluency", "validation_fold"]

assert len(data) == len(prompt) == 2000
assert not data.duplicated(KEYS).any()
assert not prompt.duplicated(KEYS).any()
assert prompt[features].notna().all().all()
assert np.isfinite(prompt[features].to_numpy(dtype=float)).all()
assert (prompt[features] >= 0).all().all()

aligned = data[KEYS + ["fluency"] + features].merge(
    prompt[KEYS + features],
    on=KEYS,
    how="inner",
    validate="one_to_one",
    suffixes=("_model", "_source")
)

assert len(aligned) == 2000

for feature in features:
    assert np.allclose(
        aligned[f"{feature}_model"],
        aligned[f"{feature}_source"],
        rtol=0, atol=1e-10
    )

# Original frozen OOF predictions, never retrained.
base = None

for name, (filename, pred_col) in MODELS.items():
    path = R / filename
    assert sha(path) == MODEL_HASHES[name]

    frame = read(path)
    assert len(frame) == 2000

    if base is None:
        base = frame[FULL + [pred_col]].copy()
    else:
        base = base.merge(
            frame[FULL + [pred_col]],
            on=FULL,
            how="inner",
            validate="one_to_one"
        )

assert len(base) == 2000

base = base.merge(
    data[KEYS + features],
    on=KEYS,
    how="inner",
    validate="one_to_one"
)

assert len(base) == 2000
assert base["audio_id"].nunique() == 400
assert base["speaker"].nunique() == 20

for name, (_, col) in MODELS.items():
    base[f"{name}_ae"] = (base[col] - base["fluency"]).abs()
    base[f"{name}_se"] = (base[col] - base["fluency"]) ** 2

# Each audio is represented by five hypotheses.
agg = {}
for f in features:
    agg[f"{f}_mean"] = (f, "mean")
    agg[f"{f}_max"] = (f, "max")
    agg[f"{f}_range"] = (
        f, lambda x: x.max() - x.min()
    )

for name, (_, col) in MODELS.items():
    agg[f"{name}_ae"] = (f"{name}_ae", "mean")
    agg[f"{name}_se"] = (f"{name}_se", "mean")
    agg[f"{name}_range"] = (
        col, lambda x: x.max() - x.min()
    )

audio = base.groupby(
    IDS + ["fluency", "validation_fold"],
    as_index=False
).agg(**agg)

assert len(audio) == 400
assert audio["audio_id"].is_unique

# Audio-level paired error differences.
audio["B2_minus_B1_AE"] = (
    audio["B2_ae"] - audio["B1_ae"]
)
audio["C2_minus_C1_AE"] = (
    audio["C2_ae"] - audio["C1_ae"]
)
audio["B2_minus_B1_range"] = (
    audio["B2_range"] - audio["B1_range"]
)
audio["C2_minus_C1_range"] = (
    audio["C2_range"] - audio["C1_range"]
)

word_feature = "prompt_asr_word_disagreement"
char_feature = "prompt_asr_char_disagreement"

assert word_feature in features
assert char_feature in features

# Quartiles are DESCRIPTIVE strata defined on these 400
# audio means; no model fitting or threshold optimization.
audio["word_disagreement_quartile"] = pd.qcut(
    audio[f"{word_feature}_mean"].rank(method="first"),
    q=4,
    labels=["Q1", "Q2", "Q3", "Q4"]
)

strata = audio.groupby(
    "word_disagreement_quartile",
    observed=True
).agg(
    audios=("audio_id", "count"),
    speakers=("speaker", "nunique"),
    mean_word_disagreement=(
        f"{word_feature}_mean", "mean"
    ),
    mean_char_disagreement=(
        f"{char_feature}_mean", "mean"
    ),
    B2_minus_B1_MAE=("B2_minus_B1_AE", "mean"),
    C2_minus_C1_MAE=("C2_minus_C1_AE", "mean"),
    B2_minus_B1_mean_range=(
        "B2_minus_B1_range", "mean"
    ),
    C2_minus_C1_mean_range=(
        "C2_minus_C1_range", "mean"
    )
).reset_index()

assert len(strata) == 4
assert strata["audios"].sum() == 400

descriptive = {
    "word_disagreement_mean": float(base[word_feature].mean()),
    "word_disagreement_median": float(base[word_feature].median()),
    "word_disagreement_max": float(base[word_feature].max()),
    "char_disagreement_mean": float(base[char_feature].mean()),
    "char_disagreement_median": float(base[char_feature].median()),
    "char_disagreement_max": float(base[char_feature].max()),
    "exact_prompt_matches": int(
        ((base[word_feature] == 0) &
         (base[char_feature] == 0)).sum()
    ),
    "audio_word_disagreement_variable": int(
        (audio[f"{word_feature}_range"] > 1e-12).sum()
    ),
    "audio_char_disagreement_variable": int(
        (audio[f"{char_feature}_range"] > 1e-12).sum()
    )
}

assert descriptive["audio_word_disagreement_variable"] == 131
assert descriptive["audio_char_disagreement_variable"] == 159

primary = {}
for name in MODELS:
    primary[name] = {
        "MAE": float(audio[f"{name}_ae"].mean()),
        "RMSE": float(np.sqrt(audio[f"{name}_se"].mean())),
        "mean_range": float(audio[f"{name}_range"].mean())
    }

expected_mae = {
    "B1": 1.2426, "B2": 1.1761,
    "C1": 0.8972, "C2": 0.7782
}
for name, value in expected_mae.items():
    assert np.isclose(
        primary[name]["MAE"], value, atol=1e-4
    )

OUT.mkdir(parents=True, exist_ok=True)

AUDIO_OUT = OUT / "step07_04_prompt_audit_per_audio.csv"
STRATA_OUT = OUT / "step07_04_prompt_disagreement_strata.csv"
REPORT_OUT = OUT / "step07_04_prompt_validity_audit.json"

def safe_csv(frame, path):
    temp = path.with_suffix(".tmp")
    frame.to_csv(
        temp, index=False, encoding="utf-8",
        float_format="%.12g"
    )
    if path.exists():
        assert sha(path) == sha(temp), (
            f"Existing output differs: {path}"
        )
        temp.unlink()
    else:
        temp.replace(path)

safe_csv(audio, AUDIO_OUT)
safe_csv(strata, STRATA_OUT)

report = {
    "status": "PASS",
    "step": "7.4",
    "analysis": "prompt_feature_validity_and_ablation",
    "original_audio_count": 400,
    "hypothesis_count": 2000,
    "speaker_count": 20,
    "feature_names": features,
    "feature_values_match_original_source": True,
    "descriptive_statistics": descriptive,
    "original_model_metrics": primary,
    "observational_only": True,
    "model_retraining": False,
    "ground_truth_wer_or_cer": False,
    "reading_prompt_is_verified_transcript": False,
    "operational_prompt_availability_verified": False,
    "quartiles": "equal_audio_count_descriptive",
    "source_sha256": {
        str(path.relative_to(P)): sha(path)
        for path in HASHES
    },
    "output_sha256": {
        AUDIO_OUT.name: sha(AUDIO_OUT),
        STRATA_OUT.name: sha(STRATA_OUT)
    },
    "limitations": [
        "Disagreement with a reading prompt is not ASR WER/CER.",
        "Deviations may reflect learner speech or ASR errors.",
        "Prompt feature availability in deployment is unverified.",
        "Quartile comparisons are descriptive, not causal.",
        "Feature-group gains are not isolated causal contributions.",
        "All five ASR hypotheses share one human label."
    ]
}

if REPORT_OUT.exists():
    assert json.loads(
        REPORT_OUT.read_text(encoding="utf-8")
    ) == report
else:
    temp = REPORT_OUT.with_suffix(".tmp")
    temp.write_text(
        json.dumps(report, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    temp.replace(REPORT_OUT)

progress = json.loads(
    CHECK.read_text(encoding="utf-8")
)
progress["step7_04_prompt_validity_audit"] = "qc_pass"
progress["step7_04_prompt_validity_sha256"] = sha(REPORT_OUT)
progress["next_task"] = (
    "Review prompt-aware feature interpretation "
    "and assess statistical claim boundaries in STEP 7.5"
)
progress["updated_at"] = datetime.now(
    timezone.utc
).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== STEP 7.4 PROMPT FEATURE VALIDITY AUDIT ===")
print("Status: PASS")
print("Original observations retained: YES")
print("Model retraining: NO")
print("Ground-truth WER/CER: NO")

print("\n=== PROMPT DISAGREEMENT DISTRIBUTION ===")
for key, value in descriptive.items():
    print(f"{key}: {value:.4f}"
          if isinstance(value, float) else f"{key}: {value}")

print("\n=== ORIGINAL MODEL METRICS ===")
for name, value in primary.items():
    print(
        f"{name}: MAE={value['MAE']:.4f}, "
        f"RMSE={value['RMSE']:.4f}, "
        f"mean_range={value['mean_range']:.4f}"
    )

print("\n=== PROMPT DISAGREEMENT QUARTILES ===")
print(strata.round(4).to_string(index=False))

print("\n=== IMPORTANT INTERPRETATION ===")
print("Reading prompt != verified human transcript")
print("Prompt disagreement != ground-truth WER/CER")
print("Operational prompt availability: NOT VERIFIED")
print("Feature ablations: OBSERVATIONAL")

print("\nReport:", REPORT_OUT)
print("SHA256:", sha(REPORT_OUT))
print("Audio CSV:", AUDIO_OUT)
print("Quartile CSV:", STRATA_OUT)
print("Next: STEP 7.5 statistical claims audit")


In [ ]:

import json
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd

P = Path("/kaggle/working/acoustic_linguistic_research")
R = P / "results/step05"
OUT = P / "results/step07_methodology"
OUT.mkdir(parents=True, exist_ok=True)

BOOT = R / "speaker_cluster_bootstrap_10000_ci.csv"
REG = OUT / "step07_03_regularization_sensitivity.csv"
PROMPT = OUT / "step07_04_prompt_disagreement_strata.csv"
CHECK = P / "checkpoints/research_progress.json"

CLAIMS = OUT / "step07_05_statistical_claims.csv"
REPORT = OUT / "step07_05_statistical_claims_audit.json"

EXPECTED = {
    BOOT: "5ef3d6dd01eb05758ccb8b4f77293396a81e988b9b9e33152cb666c6b48d1d93",
    PROMPT: "2d1f90d7d6d87a82a181bba300c14bee14e9c468ce0dd98b5a37877ab3709326"
}

def sha(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

assert sha(BOOT) == EXPECTED[BOOT]

# PROMPT hash above identifies the audit JSON, not the CSV.
PROMPT_REPORT = OUT / "step07_04_prompt_validity_audit.json"
assert sha(PROMPT_REPORT) == EXPECTED[PROMPT_REPORT] if (
    PROMPT_REPORT in EXPECTED
) else sha(PROMPT_REPORT) == EXPECTED[PROMPT]

prompt_report = json.loads(
    PROMPT_REPORT.read_text(encoding="utf-8")
)
assert prompt_report["status"] == "PASS"
assert sha(PROMPT) == prompt_report["output_sha256"][PROMPT.name]

reg_report_path = OUT / "step07_03_regularization_fairness.json"
reg_report = json.loads(
    reg_report_path.read_text(encoding="utf-8")
)
assert reg_report["status"] == "PASS"
assert sha(REG) == reg_report["output_hashes"][REG.name]

boot = pd.read_csv(BOOT)
reg = pd.read_csv(REG)
strata = pd.read_csv(PROMPT)

assert len(boot) == 33
assert len(reg) == 8
assert len(strata) == 4

rows = []

COMPARISONS = [
    ("C1_minus_A", "MAE"),
    ("C2_minus_A", "MAE"),
    ("C2_minus_A", "RMSE"),
    ("C2_minus_C1", "MAE"),
    ("C2_minus_C1", "RMSE"),
    ("C2_minus_C1", "mean_range"),
    ("B2_minus_B1", "MAE"),
    ("B2_minus_B1", "mean_range"),
    ("C1_minus_B1", "MAE"),
    ("C2_minus_B2", "MAE")
]

for comparison, metric in COMPARISONS:
    matches = boot[
        boot["comparison"].eq(comparison) &
        boot["metric"].eq(metric)
    ]
    assert len(matches) == 1
    r = matches.iloc[0]

    lower = float(r["ci_2_5"])
    upper = float(r["ci_97_5"])
    diff = float(r["observed"])

    status = (
        "CI_excludes_zero"
        if lower > 0 or upper < 0
        else "CI_includes_zero"
    )

    rows.append({
        "claim": f"{comparison}_{metric}",
        "analysis": "original_fixed_alpha_1",
        "metric": metric,
        "observed_difference": diff,
        "CI_low": lower,
        "CI_high": upper,
        "evidence_status": status,
        "interpretation_scope": (
            "Paired 20-speaker bootstrap; "
            "conditional on frozen OOF predictions"
        )
    })

# Alpha=5 is secondary methodological sensitivity.
alpha5 = reg[reg["alpha"].eq(5.0)].set_index("model")
assert set(alpha5.index) == {"B1", "B2", "C1", "C2"}

for left, right in [("C2", "C1"), ("B2", "B1")]:
    for metric in ["MAE", "RMSE", "mean_prediction_range"]:
        difference = float(
            alpha5.loc[left, metric] -
            alpha5.loc[right, metric]
        )

        rows.append({
            "claim": f"{left}_minus_{right}_{metric}",
            "analysis": "alpha_5_sensitivity",
            "metric": metric,
            "observed_difference": difference,
            "CI_low": np.nan,
            "CI_high": np.nan,
            "evidence_status": "descriptive_only",
            "interpretation_scope": (
                "Secondary alpha normalization check; "
                "no speaker bootstrap CI computed"
            )
        })

# Prompt quartiles are descriptive post-hoc strata.
for _, row in strata.iterrows():
    rows.append({
        "claim": f"{row['word_disagreement_quartile']}_C2_minus_C1_MAE",
        "analysis": "prompt_quartile",
        "metric": "MAE",
        "observed_difference": float(
            row["C2_minus_C1_MAE"]
        ),
        "CI_low": np.nan,
        "CI_high": np.nan,
        "evidence_status": "descriptive_only",
        "interpretation_scope": (
            "Quartiles of observed audio-level "
            "prompt disagreement; not causal"
        )
    })

claims = pd.DataFrame(rows)
assert len(claims) == 20

# Validate selected core claim direction and CI status.
core = claims.set_index("claim")

assert (
    core.loc["C2_minus_A_MAE", "evidence_status"]
    == "CI_excludes_zero"
)
assert core.loc["C2_minus_A_MAE", "CI_high"] < 0

assert (
    core.loc["C2_minus_A_RMSE", "evidence_status"]
    == "CI_includes_zero"
)

assert (
    core.loc["C2_minus_C1_mean_range", "evidence_status"]
    == "CI_excludes_zero"
)
assert core.loc["C2_minus_C1_mean_range", "CI_low"] > 0

temp = CLAIMS.with_suffix(".tmp")
claims.to_csv(
    temp, index=False, encoding="utf-8",
    float_format="%.12g"
)

if CLAIMS.exists():
    assert sha(CLAIMS) == sha(temp)
    temp.unlink()
else:
    temp.replace(CLAIMS)

report = {
    "status": "PASS",
    "step": "7.5",
    "analysis": "statistical_claims_and_generalization_audit",
    "original_audio_count": 400,
    "independent_speaker_clusters": 20,
    "asr_hypothesis_rows": 2000,
    "bootstrap_iterations": 10000,
    "bootstrap_refitting": False,
    "multiple_comparison_adjustment": False,
    "formal_familywise_significance_claims": False,
    "alpha5_CI_available": False,
    "official_test_evaluated": False,
    "claim_count": len(claims),
    "source_sha256": {
        "bootstrap": sha(BOOT),
        "regularization": sha(REG),
        "prompt_strata": sha(PROMPT)
    },
    "claims_sha256": sha(CLAIMS),
    "approved_interpretation": [
        "C2 shows lower observed MAE than A and C1.",
        "The paired speaker-bootstrap MAE intervals exclude zero.",
        "C2 does not demonstrate an RMSE advantage over A "
        "in the original alpha=1 evaluation.",
        "C2 has greater mean prediction range than C1.",
        "Alpha=5 supports descriptive regularization robustness.",
        "Prompt disagreement does not measure verified ASR WER/CER."
    ],
    "limitations": [
        "Only 20 speaker clusters.",
        "Bootstrap conditional on fixed OOF predictions.",
        "Multiple comparisons were not multiplicity-adjusted.",
        "No external dataset validation.",
        "Official test partition remains sealed.",
        "Post-hoc model and subset comparisons are exploratory.",
        "Prompt feature deployment availability is not established."
    ]
}

if REPORT.exists():
    assert json.loads(
        REPORT.read_text(encoding="utf-8")
    ) == report
else:
    temp = REPORT.with_suffix(".tmp")
    temp.write_text(
        json.dumps(report, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    temp.replace(REPORT)

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step7_05_statistical_claims"] = "qc_pass"
progress["step7_05_statistical_claims_sha256"] = sha(CLAIMS)
progress["next_task"] = (
    "STEP 7.6 reviewer readiness and final "
    "methodological decisions"
)
CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== STEP 7.5 STATISTICAL CLAIMS AUDIT ===")
print("Status: PASS")
print("Claim records:", len(claims))
print("Speaker clusters:", 20)
print("Multiple-testing adjustment: NO")
print("Official test evaluated: NO")

print("\n=== ORIGINAL MODEL CLAIMS ===")
print(
    claims[
        claims["analysis"].eq("original_fixed_alpha_1")
    ][
        ["claim", "observed_difference",
         "CI_low", "CI_high", "evidence_status"]
    ].round(4).to_string(index=False)
)

print("\n=== SECONDARY ALPHA=5 CLAIMS ===")
print(
    claims[
        claims["analysis"].eq("alpha_5_sensitivity")
    ][
        ["claim", "observed_difference",
         "evidence_status"]
    ].round(4).to_string(index=False)
)

print("\n=== PROMPT QUARTILE CLAIMS ===")
print(
    claims[
        claims["analysis"].eq("prompt_quartile")
    ][
        ["claim", "observed_difference",
         "evidence_status"]
    ].round(4).to_string(index=False)
)

print("\n=== LIMITATIONS ===")
for item in report["limitations"]:
    print("-", item)

print("\nClaims CSV:", CLAIMS)
print("Report:", REPORT)
print("Report SHA256:", sha(REPORT))
print("Next: STEP 7.6 reviewer readiness")


In [ ]:

import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd

P = Path("/kaggle/working/acoustic_linguistic_research")
OUT = P / "results/step07_methodology"
CHECK = P / "checkpoints/research_progress.json"

SOURCES = {
    "7.1": OUT / "step07_01_dataset_integrity.json",
    "7.2": OUT / "step07_02_leakage_audit.json",
    "7.3": OUT / "step07_03_regularization_fairness.json",
    "7.4": OUT / "step07_04_prompt_validity_audit.json",
    "7.5": OUT / "step07_05_statistical_claims_audit.json",
}

KNOWN_HASHES = {
    "7.1": "9b48a723f15fc080c86e23a9e3b61b6183917818de4badc79770080a46eb8a31",
    "7.2": "d0deb85ae53f034664f870132913cea3498b26a22caed7c042a66e8a41d900d0",
    "7.3": "7a10748ff0e91cd199ea0b5242aae28da1f7f7aba5deeec03f86ae00b14a701f",
    "7.4": "2d1f90d7d6d87a82a181bba300c14bee14e9c468ce0dd98b5a37877ab3709326",
    "7.5": "f0b9f8aedcf1e72ea467cff21bde9d33836b43bde3c5a81cb8a0863f86ad2f8a"
}

REGISTRY = OUT / "step07_06_reviewer_decision_registry.csv"
REPORT = OUT / "step07_06_reviewer_readiness.json"

def sha(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

audits = {}

for step, path in SOURCES.items():
    assert path.is_file(), f"Missing: {path}"
    assert sha(path) == KNOWN_HASHES[step], (
        f"SHA mismatch: STEP {step}"
    )

    audits[step] = json.loads(
        path.read_text(encoding="utf-8")
    )
    assert audits[step]["status"] == "PASS"

assert audits["7.1"]["speaker_disjoint"]
assert audits["7.2"]["speaker_disjoint"]
assert audits["7.2"]["audio_disjoint"]
assert audits["7.2"]["imputation_fit_on_training_fold"]
assert audits["7.2"]["scaler_fit_on_training_fold"]

assert (
    audits["7.3"]["primary_original_predictions_unchanged"]
)
assert audits["7.4"]["ground_truth_wer_or_cer"] is False
assert audits["7.5"]["multiple_comparison_adjustment"] is False

# These are methodological review decisions,
# not new experimentally verified findings.
DECISIONS = [
    (
        "D01", "Dataset integrity", "verified",
        "Use 400 audios, 20 speakers and frozen splits.",
        "7.1", "Methods"
    ),
    (
        "D02", "Downstream leakage controls", "verified",
        "Report speaker-disjoint CV and train-only preprocessing.",
        "7.2", "Methods"
    ),
    (
        "D03", "OOF model reproducibility", "verified",
        "Retain saved predictions and reconstruction evidence.",
        "7.2", "Supplementary"
    ),
    (
        "D04", "Regularization normalization", "verified_with_scope",
        "Report alpha=1 primary and alpha=5 sensitivity; "
        "do not select alpha based on observed validation results.",
        "7.3", "Methods and Supplementary"
    ),
    (
        "D05", "Prompt-aware model interpretation",
        "verified_with_scope",
        "Treat B2/C2 as prompt-conditional secondary models; "
        "avoid ground-truth WER/CER terminology.",
        "7.4", "Methods and Discussion"
    ),
    (
        "D06", "Bootstrap uncertainty", "verified_with_scope",
        "Report paired speaker-cluster intervals; "
        "state no model refitting or multiplicity correction.",
        "7.5", "Results and Limitations"
    ),
    (
        "D07", "Original sample selection provenance",
        "requires_documentation",
        "Locate and preserve initial 20-speaker sampling code, "
        "selection criteria and random seed, if used.",
        "7.1", "Methods and Reproducibility"
    ),
    (
        "D08", "ASR decoding reproducibility",
        "requires_documentation",
        "Locate exact model revision and H1-H5 decoding "
        "parameters from original notebooks and configs.",
        "STEP 2", "Methods and Supplementary"
    ),
    (
        "D09", "ASR pretraining overlap",
        "unverified_external_limit",
        "Do not assert absence of overlap with ASR pretraining.",
        "7.2", "Limitations"
    ),
    (
        "D10", "Prompt availability in deployment",
        "requires_scope_statement",
        "Define the application as read-aloud scoring where "
        "the reading prompt is accessible when using B2/C2.",
        "7.4", "Methods and Discussion"
    ),
    (
        "D11", "External validation",
        "not_performed",
        "Describe conclusions as pilot results; "
        "do not generalize to other datasets or languages.",
        "7.5", "Limitations"
    ),
    (
        "D12", "Multiple statistical comparisons",
        "verified_with_scope",
        "Present intervals descriptively rather than claiming "
        "familywise-controlled statistical significance.",
        "7.5", "Results and Limitations"
    ),
    (
        "D13", "Anomaly exclusions",
        "verified_with_scope",
        "Use all 400 audios in primary analyses; "
        "report 399/396 results as post-hoc sensitivity.",
        "STEP 6.2", "Results and Supplementary"
    ),
    (
        "D14", "Primary research emphasis",
        "methodological_decision",
        "Present A/B1/C1 as prompt-independent primary "
        "comparisons and B2/C2 as prompt-aware secondary "
        "ablations; avoid post-hoc winner selection.",
        "7.4 and 7.5", "Article structure"
    )
]

registry = pd.DataFrame(
    DECISIONS,
    columns=[
        "decision_id", "issue", "status",
        "required_action_or_reporting_rule",
        "evidence", "manuscript_section"
    ]
)

assert len(registry) == 14
assert registry["decision_id"].is_unique

unresolved_statuses = {
    "requires_documentation",
    "requires_scope_statement",
    "unverified_external_limit",
    "not_performed"
}

unresolved = registry[
    registry["status"].isin(unresolved_statuses)
]

# Audit is complete, but submission readiness
# remains conditional on unresolved documentation.
readiness = "CONDITIONAL_NOT_SUBMISSION_READY"

OUT.mkdir(parents=True, exist_ok=True)

def safe_csv(frame, path):
    tmp = path.with_suffix(".tmp")
    frame.to_csv(
        tmp, index=False, encoding="utf-8"
    )

    if path.exists():
        assert sha(path) == sha(tmp), (
            f"Existing registry differs: {path}"
        )
        tmp.unlink()
    else:
        tmp.replace(path)

safe_csv(registry, REGISTRY)

report = {
    "status": "PASS",
    "step": "7.6",
    "analysis": "reviewer_readiness_and_method_decisions",
    "methodological_audits_completed": list(SOURCES),
    "research_sample": {
        "speakers": 20,
        "audios": 400,
        "asr_hypotheses": 2000
    },
    "decision_count": len(registry),
    "unresolved_or_limited_items": unresolved[
        "decision_id"
    ].tolist(),
    "manuscript_readiness": readiness,
    "model_retraining": False,
    "official_test_used_in_this_cell": False,
    "original_results_changed": False,
    "source_report_sha256": {
        step: sha(path)
        for step, path in SOURCES.items()
    },
    "registry_sha256": sha(REGISTRY),
    "next_recommended_action": (
        "Resolve sample-selection and ASR-decoding "
        "documentation before final Methods drafting; "
        "prepare figures, literature review and manuscript."
    )
}

if REPORT.exists():
    assert json.loads(
        REPORT.read_text(encoding="utf-8")
    ) == report, "Existing readiness report differs"
else:
    tmp = REPORT.with_suffix(".tmp")
    tmp.write_text(
        json.dumps(
            report, indent=2, ensure_ascii=False
        ),
        encoding="utf-8"
    )
    tmp.replace(REPORT)

progress = json.loads(
    CHECK.read_text(encoding="utf-8")
)
progress["step7_06_reviewer_readiness"] = "qc_pass"
progress["step7_06_registry_sha256"] = sha(REGISTRY)
progress["step7_methodology_status"] = "completed_with_open_items"
progress["next_task"] = (
    "Archive STEP 7 and resolve sampling/ASR "
    "provenance before manuscript finalization"
)
progress["updated_at"] = datetime.now(
    timezone.utc
).isoformat()

CHECK.write_text(
    json.dumps(
        progress, indent=2, ensure_ascii=False
    ),
    encoding="utf-8"
)

print("=== STEP 7.6 REVIEWER READINESS ===")
print("Status: PASS")
print("STEP 7.1-7.5 source audit checks: PASS")
print("Decision records:", len(registry))
print("Manuscript readiness:", readiness)

print("\n=== REVIEWER DECISION REGISTRY ===")
print(
    registry[
        ["decision_id", "issue", "status"]
    ].to_string(index=False)
)

print("\n=== UNRESOLVED OR LIMITED ITEMS ===")
print(
    unresolved[
        [
            "decision_id", "issue",
            "required_action_or_reporting_rule"
        ]
    ].to_string(index=False)
)

print("\nRegistry:", REGISTRY)
print("Registry SHA256:", sha(REGISTRY))
print("Report:", REPORT)
print("Report SHA256:", sha(REPORT))
print("Checkpoint updated.")


In [ ]:

from pathlib import Path
import hashlib
import json
import re

import pandas as pd

P = Path("/kaggle/working/acoustic_linguistic_research")
OUT = P / "results/step08_provenance"
OUT.mkdir(parents=True, exist_ok=True)

INVENTORY = OUT / "step08_01_provenance_inventory.csv"
REPORT = OUT / "step08_01_provenance_inventory.json"

EXTENSIONS = {".json", ".py", ".ipynb", ".yaml", ".yml", ".toml", ".txt", ".md"}
MAX_FILE_SIZE = 5 * 1024 * 1024

PATTERNS = {
    "sampling": re.compile(
        r"sample|speaker|selection|random_state|"
        r"random.seed|default_rng|pilot_index|"
        r"train_test_split|stratified",
        re.IGNORECASE
    ),
    "asr": re.compile(
        r"whisper|decode|generate|num_beams|"
        r"temperature|top_k|top_p|"
        r"do_sample|language|task|"
        r"return_timestamps|forced_decoder_ids|"
        r"model_revision|model_id|"
        r"repetition_penalty|no_repeat_ngram",
        re.IGNORECASE
    ),
    "revision": re.compile(
        r"06385584fad212b26134c656fdd3ccf9f093f33e|"
        r"e8727524f962ee844a7319d92be39ac1bd25655a|"
        r"revision|commit_hash|seed",
        re.IGNORECASE
    )
}

def sha256(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def safe_text(path):
    try:
        return path.read_text(encoding="utf-8")
    except UnicodeDecodeError:
        return None

files = []

for root in [P, Path("/kaggle/working")]:
    for path in root.glob("*") if root != P else root.rglob("*"):
        if not path.is_file():
            continue
        if path.suffix.lower() not in EXTENSIONS:
            continue
        if path.stat().st_size > MAX_FILE_SIZE:
            continue
        if ".ipynb_checkpoints" in path.parts:
            continue
        files.append(path)

files = sorted(set(files))

records = []
searched_count = 0

for path in files:
    raw = safe_text(path)
    if raw is None:
        continue

    searched_count += 1

    if path.suffix.lower() == ".ipynb":
        try:
            notebook = json.loads(raw)
            parts = []
            for idx, cell in enumerate(notebook.get("cells", [])):
                if cell.get("cell_type") != "code":
                    continue
                source = cell.get("source", [])
                if isinstance(source, list):
                    source = "".join(source)
                parts.append((f"code_cell_{idx}", source))
        except Exception:
            continue
    else:
        parts = [("file", raw)]

    for section, content in parts:
        lines = content.splitlines()

        for line_number, line in enumerate(lines, 1):
            matched = [
                name for name, pattern in PATTERNS.items()
                if pattern.search(line)
            ]

            if not matched:
                continue

            # Restrict to relevant snippets without losing
            # source file and line references.
            records.append({
                "file": str(path),
                "section": section,
                "line": line_number,
                "categories": ",".join(matched),
                "snippet": line.strip()[:350],
                "file_sha256": sha256(path)
            })

inventory = pd.DataFrame(
    records,
    columns=[
        "file", "section", "line",
        "categories", "snippet", "file_sha256"
    ]
)

inventory.to_csv(INVENTORY, index=False, encoding="utf-8")

# Prioritize configurations and code rather than
# interpreting keyword occurrences as proof.
priority = inventory[
    inventory["snippet"].str.contains(
        r"num_beams|do_sample|temperature|"
        r"random_state|default_rng|"
        r"from_pretrained|revision=|"
        r"speaker.*sample|sample.*speaker",
        case=False,
        regex=True,
        na=False
    )
].copy()

priority = priority.head(60)

report = {
    "status": "INVENTORY_COMPLETED",
    "purpose": "Locate D07 and D08 documentary evidence",
    "files_searched": searched_count,
    "matching_lines": len(inventory),
    "priority_lines": len(priority),
    "sampling_verified": False,
    "asr_configuration_verified": False,
    "no_models_retrained": True,
    "no_asr_inference": True,
    "inventory_sha256": sha256(INVENTORY),
    "caution": (
        "Keyword matches are leads, not validated provenance. "
        "Source code and configurations must be inspected "
        "before D07 or D08 can be marked resolved."
    )
}

REPORT.write_text(
    json.dumps(report, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== STEP 8.1 PROVENANCE INVENTORY ===")
print("Status:", report["status"])
print("Files searched:", searched_count)
print("Matching lines:", len(inventory))
print("Sampling verified: NO")
print("ASR configuration verified: NO")

print("\n=== MATCHES BY CATEGORY ===")
for category in PATTERNS:
    count = inventory["categories"].str.contains(
        category, regex=False
    ).sum()
    print(category, ":", int(count))

print("\n=== PRIORITY SOURCE EVIDENCE ===")
if len(priority):
    print(
        priority[
            ["file", "section", "line", "snippet"]
        ].to_string(index=False, max_colwidth=110)
    )
else:
    print("No priority source lines found.")

print("\n=== SOURCE FILES WITH MOST MATCHES ===")
if len(inventory):
    print(
        inventory["file"]
        .value_counts()
        .head(15)
        .to_string()
    )

print("\nInventory:", INVENTORY)
print("Inventory SHA256:", sha256(INVENTORY))
print("Report:", REPORT)
print("Next: inspect actual source evidence for D07 and D08")


In [ ]:

from pathlib import Path
import hashlib
import json

P = Path("/kaggle/working/acoustic_linguistic_research")

SOURCES = {
    "D07_step01_report": P / "reports/step01_report.md",
    "D07_split_candidate_qc": P / "reports/step05_speaker_split_candidate_qc.json",
    "D07_speaker_independence": P / "reports/speaker_independence_report.json",
    "D07_frozen_cv": P / "configs/speaker_disjoint_cv_v1.json",
    "D08_asr_config": P / "configs/asr_hypothesis_pilot_v1.json",
    "D08_asr_freeze": P / "reports/step03_asr_configuration_freeze.json",
    "D08_whisper_revision": P / "reports/step03_whisper_model_revision.json",
    "D08_beam_test": P / "reports/step03_whisper_beam_test.json"
}

def sha256(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

evidence = {}

for label, path in SOURCES.items():
    print("\n" + "=" * 65)
    print("SOURCE:", label)
    print("FILE:", path)

    if not path.is_file():
        print("STATUS: MISSING")
        evidence[label] = {"status": "MISSING"}
        continue

    digest = sha256(path)
    print("SHA256:", digest)

    text = path.read_text(encoding="utf-8")
    print("SIZE:", len(text), "characters")

    if path.suffix.lower() == ".json":
        obj = json.loads(text)
        formatted = json.dumps(
            obj,
            indent=2,
            ensure_ascii=False
        )
    else:
        formatted = text

    # Avoid losing important data to notebook output truncation.
    if len(formatted) > 10000:
        print(formatted[:10000])
        print(
            "\n[DISPLAY TRUNCATED — original source unchanged]"
        )
    else:
        print(formatted)

    evidence[label] = {
        "status": "FOUND",
        "file": str(path),
        "sha256": digest,
        "size_bytes": path.stat().st_size
    }

print("\n" + "=" * 65)
print("=== STEP 8.2 ORIGINAL PROVENANCE EVIDENCE ===")
print("Sources inspected:", len(SOURCES))
print(
    "Found:",
    sum(v["status"] == "FOUND" for v in evidence.values())
)
print(
    "Missing:",
    sum(v["status"] == "MISSING" for v in evidence.values())
)

print("\n=== SOURCE HASH SUMMARY ===")
for label, item in evidence.items():
    print(
        label,
        "|",
        item["status"],
        "|",
        item.get("sha256", "-")
    )

print("\nD07 status: EVIDENCE UNDER REVIEW")
print("D08 status: EVIDENCE UNDER REVIEW")
print("No files modified.")


In [ ]:

from pathlib import Path
import pandas as pd
import json, hashlib, re

P = Path("/kaggle/working/acoustic_linguistic_research")
OUT = P / "results/step08_provenance"
OUT.mkdir(parents=True, exist_ok=True)

CFG = P / "configs/asr_hypothesis_pilot_v1.json"
FREEZE = P / "reports/step03_asr_configuration_freeze.json"
MODEL = P / "reports/step03_whisper_model_revision.json"

def sha(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for b in iter(lambda: f.read(1024*1024), b""):
            h.update(b)
    return h.hexdigest()

cfg = json.loads(CFG.read_text(encoding="utf-8"))
freeze = json.loads(FREEZE.read_text(encoding="utf-8"))
model = json.loads(MODEL.read_text(encoding="utf-8"))

assert cfg["hypotheses"] == freeze["hypotheses"]
assert cfg["model_id"] == freeze["model_id"] == model["model_id"]
assert (
    cfg["model_revision"]
    == freeze["model_revision"]
    == model["model_revision"]
)
assert freeze["configuration_sha256"] == sha(CFG)

print("=== STEP 8.3 FROZEN ASR CROSSCHECK ===")
print("Config and freeze consistent: PASS")
print("Model revision consistent: PASS")
print("Frozen config SHA256 consistent: PASS")
print("Full 400-audio inference code audited: NOT YET")

# Look for potential original sample selection documents.
keywords = re.compile(
    r"sample|select|pilot|speaker|asr|hypothes|"
    r"manifest|progress|inference|batch",
    re.I
)

candidates = sorted(
    path for path in P.rglob("*")
    if path.is_file()
    and keywords.search(path.name)
    and path.suffix.lower() in {
        ".json", ".csv", ".py", ".ipynb", ".md", ".txt"
    }
)

records = []
for path in candidates:
    records.append({
        "path": str(path.relative_to(P)),
        "type": path.suffix.lower(),
        "size_bytes": path.stat().st_size,
        "sha256": sha(path)
    })

inventory = pd.DataFrame(records)
inventory_path = OUT / "step08_03_candidate_files.csv"
inventory.to_csv(inventory_path, index=False)

print("\n=== POTENTIAL PROVENANCE FILES ===")
print(inventory[["path", "type", "size_bytes"]].to_string(
    index=False, max_rows=90, max_colwidth=95
))

# Inspect JSON keys and relevant nested settings; do not
# assume that keyword matches prove a sampling procedure.
relevant_keys = re.compile(
    r"sample|select|seed|random|speaker|revision|"
    r"model|decode|hypothes|inference|"
    r"generation|batch|pilot|source",
    re.I
)

evidence = []

def inspect(value, prefix="", depth=0):
    if depth > 5:
        return
    if isinstance(value, dict):
        for key, item in value.items():
            full = f"{prefix}.{key}" if prefix else str(key)
            if relevant_keys.search(str(key)):
                if isinstance(item, (str, int, float, bool)):
                    evidence.append((source_name, full, str(item)[:180]))
                elif isinstance(item, list):
                    evidence.append((
                        source_name, full,
                        f"list length={len(item)}"
                    ))
            inspect(item, full, depth + 1)
    elif isinstance(value, list):
        for i, item in enumerate(value[:12]):
            inspect(item, f"{prefix}[{i}]", depth + 1)

for path in candidates:
    if path.suffix.lower() != ".json":
        continue
    if path.stat().st_size > 1_000_000:
        continue
    try:
        obj = json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        continue
    source_name = str(path.relative_to(P))
    inspect(obj)

evidence_df = pd.DataFrame(
    evidence, columns=["file", "key", "value"]
)
evidence_path = OUT / "step08_03_metadata_evidence.csv"
evidence_df.to_csv(evidence_path, index=False)

# Focus on sample selection and full-run metadata.
focus = evidence_df[
    evidence_df["key"].str.contains(
        r"seed|random|select|sampling|sample_method|"
        r"source.*speaker|model_revision|generation|"
        r"inference|batch",
        case=False, regex=True, na=False
    )
]

print("\n=== TARGETED METADATA ===")
if focus.empty:
    print("No targeted metadata keys found.")
else:
    print(focus.head(90).to_string(
        index=False, max_colwidth=85
    ))

# Inspect existing CSV filenames and schemas only.
print("\n=== CANDIDATE CSV SCHEMAS ===")
csv_candidates = [
    path for path in candidates
    if path.suffix.lower() == ".csv"
    and any(term in path.name.lower() for term in [
        "speaker", "pilot", "sample", "asr", "hypothes"
    ])
]

for path in csv_candidates[:20]:
    try:
        head = pd.read_csv(path, nrows=3, dtype=str)
        print(str(path.relative_to(P)))
        print("  columns:", list(head.columns)[:20])
        print("  example audio IDs:",
              head["audio_id"].tolist()
              if "audio_id" in head.columns else "N/A")
    except Exception as exc:
        print(path.name, "READ ERROR:", str(exc)[:100])

report = {
    "status": "EVIDENCE_INVENTORY_COMPLETE",
    "step": "8.3",
    "frozen_config_consistency": True,
    "model_revision_consistency": True,
    "full_run_code_verified": False,
    "original_sampling_method_verified": False,
    "candidate_files": len(candidates),
    "inventory_sha256": sha(inventory_path),
    "evidence_sha256": sha(evidence_path),
    "source_config_sha256": sha(CFG),
    "no_model_retraining": True,
    "no_asr_inference": True
}

report_path = OUT / "step08_03_targeted_provenance.json"
report_path.write_text(
    json.dumps(report, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\n=== STEP 8.3 SUMMARY ===")
print("Status:", report["status"])
print("Candidate files:", len(candidates))
print("Extracted metadata records:", len(evidence_df))
print("Original sampling method verified: NO")
print("Full-run ASR code verified: NO")
print("Evidence CSV:", evidence_path)
print("Report:", report_path)


In [ ]:

from pathlib import Path
import json
import hashlib
import numpy as np
import pandas as pd

P = Path("/kaggle/working/acoustic_linguistic_research")
OUT = P / "results/step08_provenance"
OUT.mkdir(parents=True, exist_ok=True)

SPEAKER_FILE = P / "checkpoints/pilot_speaker_ids.json"
SELECTION = P / "data/metadata/pilot_selection_400.csv"
ASR = P / "features/asr/full/asr_hypotheses_0001-0400.csv"
CFG = P / "configs/asr_hypothesis_pilot_v1.json"
FREEZE = P / "reports/step03_asr_configuration_freeze.json"

REPORT = OUT / "step08_04_sampling_asr_consistency.json"
ISSUES = OUT / "step08_04_asr_metadata_mismatches.csv"

def sha(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

for path in [SPEAKER_FILE, SELECTION, ASR, CFG, FREEZE]:
    assert path.is_file(), f"Missing: {path}"

# Verify the previously frozen ASR output.
assert sha(ASR) == (
    "b67dfe5cca2329407ab07658809434c2e3bad00d227485f3f8c8c9c1dbbf3e80"
)

speaker_config = json.loads(
    SPEAKER_FILE.read_text(encoding="utf-8")
)
cfg = json.loads(CFG.read_text(encoding="utf-8"))
freeze = json.loads(FREEZE.read_text(encoding="utf-8"))

assert cfg["hypotheses"] == freeze["hypotheses"]
assert freeze["configuration_sha256"] == sha(CFG)

sel = pd.read_csv(
    SELECTION,
    dtype={"audio_id": "string", "speaker": "string"}
)
asr = pd.read_csv(
    ASR,
    dtype={"audio_id": "string", "speaker": "string"}
)

assert len(sel) == 400
assert len(asr) == 2000
assert sel["audio_id"].nunique() == 400
assert sel["speaker"].nunique() == 20

assert not sel.duplicated("audio_id").any()
assert not asr.duplicated(
    ["audio_id", "hypothesis_id"]
).any()

assert asr.groupby("audio_id")[
    "hypothesis_id"
].nunique().eq(5).all()

assert set(sel["audio_id"]) == set(asr["audio_id"])

mapping = asr[
    ["audio_id", "speaker"]
].drop_duplicates()

assert len(mapping) == 400

mapping = mapping.merge(
    sel[["audio_id", "speaker"]],
    on="audio_id",
    how="inner",
    validate="one_to_one",
    suffixes=("_asr", "_selection")
)

assert len(mapping) == 400

speaker_mapping_ok = bool(
    (mapping["speaker_asr"]
     == mapping["speaker_selection"]).all()
)

print("=== D07 SAVED SAMPLING CONFIGURATION ===")
print(json.dumps(
    speaker_config,
    indent=2,
    ensure_ascii=False
))

print("\n=== D07 SELECTION INTEGRITY ===")
print("Selected audios:", len(sel))
print("Selected speakers:", sel["speaker"].nunique())
print("Audio IDs match ASR:", True)
print("Speaker mapping match:", speaker_mapping_ok)

speaker_counts = (
    sel.groupby("speaker")["audio_id"]
    .nunique()
    .sort_index()
)
print("\n=== SELECTED SPEAKERS ===")
print(speaker_counts.to_string())
print("All speakers have 20 audios:",
      bool(speaker_counts.eq(20).all()))

print("\n=== SELECTION ORDER ===")
print(
    sel[
        ["pilot_index", "audio_id", "speaker"]
    ].head(12).to_string(index=False)
)

# Check every full-run ASR row against the
# expected hypothesis-specific decoding metadata.
expected = {
    item["id"]: item
    for item in cfg["hypotheses"]
}

assert set(expected) == {"H1", "H2", "H3", "H4", "H5"}
assert set(asr["hypothesis_id"]) == set(expected)

required_columns = [
    "hypothesis_id", "method", "model_id",
    "model_revision", "do_sample",
    "num_beams", "temperature", "seed"
]

assert set(required_columns).issubset(asr.columns)

def normalize_bool(value):
    if pd.isna(value):
        return None
    s = str(value).strip().lower()
    if s in {"true", "1", "1.0"}:
        return True
    if s in {"false", "0", "0.0"}:
        return False
    return "UNRECOGNIZED:" + s

def normalize_number(value):
    if pd.isna(value):
        return None
    try:
        return float(value)
    except (TypeError, ValueError):
        return "UNRECOGNIZED:" + str(value)

def same_value(actual, expected_value):
    if expected_value is None:
        return actual is None
    if isinstance(expected_value, (int, float)):
        return (
            isinstance(actual, (int, float))
            and not isinstance(actual, bool)
            and np.isclose(
                actual, float(expected_value),
                rtol=0, atol=1e-10
            )
        )
    return actual == expected_value

problems = []
checks = []

for hyp, group in asr.groupby("hypothesis_id"):
    spec = expected[hyp]
    mismatch_count = 0

    for _, row in group.iterrows():

        checks_for_row = {
            "method": (
                str(row["method"]).strip(),
                spec["method"]
            ),
            "model_id": (
                str(row["model_id"]).strip(),
                cfg["model_id"]
            ),
            "model_revision": (
                str(row["model_revision"]).strip(),
                cfg["model_revision"]
            ),
            "do_sample": (
                normalize_bool(row["do_sample"]),
                spec["do_sample"]
            )
        }

        # Inspect explicit or conventionally implied fields.
        # Missing values are not silently converted to
        # configured values.
        for field in ["num_beams", "temperature", "seed"]:
            actual = normalize_number(row[field])
            expected_value = spec.get(field)

            if field == "num_beams" and hyp in {"H3", "H4", "H5"}:
                expected_value = cfg[
                    "runtime_generation_rules"
                ]["num_beams_sampling"]

            checks_for_row[field] = (
                actual,
                expected_value
            )

        for field, (actual, wanted) in checks_for_row.items():
            if not same_value(actual, wanted):
                mismatch_count += 1
                problems.append({
                    "audio_id": row["audio_id"],
                    "hypothesis_id": hyp,
                    "field": field,
                    "observed": str(actual),
                    "expected": str(wanted)
                })

    checks.append({
        "hypothesis_id": hyp,
        "rows": len(group),
        "metadata_mismatches": mismatch_count
    })

checks_df = pd.DataFrame(checks)
issues_df = pd.DataFrame(
    problems,
    columns=[
        "audio_id", "hypothesis_id", "field",
        "observed", "expected"
    ]
)

issues_df.to_csv(
    ISSUES, index=False, encoding="utf-8"
)

print("\n=== D08 FULL-RUN ASR METADATA ===")
print(checks_df.to_string(index=False))

print("\n=== D08 ACTUAL FIELD VALUES ===")
for hyp, group in asr.groupby("hypothesis_id"):
    print("\n", hyp)
    for field in [
        "method", "model_id", "model_revision",
        "do_sample", "num_beams", "temperature", "seed"
    ]:
        values = (
            group[field].astype("string")
            .fillna("<missing>")
            .value_counts()
            .head(6)
            .to_dict()
        )
        print(" ", field, ":", values)

print("\n=== ASR METADATA MISMATCH EXAMPLES ===")
if len(issues_df):
    print(issues_df.head(25).to_string(index=False))
else:
    print("None")

report = {
    "step": "8.4",
    "status": (
        "METADATA_CONSISTENT"
        if len(issues_df) == 0 and speaker_mapping_ok
        else "NEEDS_REVIEW"
    ),
    "selected_audios": 400,
    "selected_speakers": 20,
    "speaker_mapping_consistent": speaker_mapping_ok,
    "all_speakers_have_20_audios": bool(
        speaker_counts.eq(20).all()
    ),
    "asr_hypothesis_rows": len(asr),
    "asr_metadata_mismatch_count": len(issues_df),
    "sampling_algorithm_verified": False,
    "full_inference_source_code_verified": False,
    "models_retrained": False,
    "source_hashes": {
        str(p.relative_to(P)): sha(p)
        for p in [SPEAKER_FILE, SELECTION, ASR, CFG, FREEZE]
    },
    "mismatch_csv_sha256": sha(ISSUES),
    "methodology_note": (
        "The audit compares saved generation metadata "
        "against the frozen configuration. It does not "
        "independently execute the original ASR code "
        "or reconstruct the original sampling algorithm."
    )
}

REPORT.write_text(
    json.dumps(
        report, indent=2, ensure_ascii=False
    ),
    encoding="utf-8"
)

print("\n=== STEP 8.4 SUMMARY ===")
print("Status:", report["status"])
print("D07 sampling algorithm independently verified: NO")
print("D08 full inference code independently verified: NO")
print("ASR metadata mismatch count:", len(issues_df))
print("Report:", REPORT)
print("Report SHA256:", sha(REPORT))
print("Mismatch CSV:", ISSUES)
print("No model retraining or ASR inference performed.")


In [ ]:

from pathlib import Path
import json
import hashlib
import math
import pandas as pd

P = Path("/kaggle/working/acoustic_linguistic_research")
OUT = P / "results/step08_provenance"
OUT.mkdir(parents=True, exist_ok=True)

ASR = P / "features/asr/full/asr_hypotheses_0001-0400.csv"
CFG = P / "configs/asr_hypothesis_pilot_v1.json"
FREEZE = P / "reports/step03_asr_configuration_freeze.json"
SPEAKERS = P / "checkpoints/pilot_speaker_ids.json"
SELECTION = P / "data/metadata/pilot_selection_400.csv"

REPORT = OUT / "step08_04_corrected_metadata_audit.json"
MISMATCH = OUT / "step08_04_corrected_metadata_mismatches.csv"
SUMMARY = OUT / "step08_04_corrected_hypothesis_summary.csv"

def sha(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

assert sha(ASR) == (
    "b67dfe5cca2329407ab07658809434c2e3bad00d227485f3f8c8c9c1dbbf3e80"
)

cfg = json.loads(CFG.read_text(encoding="utf-8"))
freeze = json.loads(FREEZE.read_text(encoding="utf-8"))
speaker_cfg = json.loads(SPEAKERS.read_text(encoding="utf-8"))

assert cfg["hypotheses"] == freeze["hypotheses"]
assert freeze["configuration_sha256"] == sha(CFG)

asr = pd.read_csv(
    ASR, dtype={"audio_id": "string", "speaker": "string"}
)
sel = pd.read_csv(
    SELECTION, dtype={"audio_id": "string", "speaker": "string"}
)

assert len(asr) == 2000
assert len(sel) == 400
assert not asr.duplicated(
    ["audio_id", "hypothesis_id"]
).any()

assert set(asr["audio_id"]) == set(sel["audio_id"])
assert set(sel["speaker"]) == set(speaker_cfg["speaker_ids"])
assert speaker_cfg["seed"] == 42

counts = sel.groupby("speaker")["audio_id"].nunique()
assert len(counts) == 20 and counts.eq(20).all()

mapping = (
    asr[["audio_id", "speaker"]]
    .drop_duplicates()
    .merge(
        sel[["audio_id", "speaker"]],
        on="audio_id",
        validate="one_to_one",
        suffixes=("_asr", "_selected")
    )
)
assert len(mapping) == 400
assert mapping["speaker_asr"].eq(
    mapping["speaker_selected"]
).all()

def parse_bool(value):
    if pd.isna(value):
        return None
    s = str(value).strip().lower()
    if s in ("true", "1", "1.0"):
        return True
    if s in ("false", "0", "0.0"):
        return False
    return "INVALID_BOOL:" + s

def parse_number(value):
    if pd.isna(value):
        return None
    try:
        x = float(value)
        return x if math.isfinite(x) else "INVALID_NUMBER"
    except (TypeError, ValueError):
        return "INVALID_NUMBER:" + str(value)

def compare(actual, expected):
    # Boolean must be checked BEFORE numeric types.
    if isinstance(expected, bool):
        return type(actual) is bool and actual == expected

    if expected is None:
        return actual is None

    if isinstance(expected, (int, float)):
        return (
            isinstance(actual, (int, float))
            and not isinstance(actual, bool)
            and math.isclose(
                float(actual), float(expected),
                rel_tol=0, abs_tol=1e-10
            )
        )

    return actual == expected

specs = {x["id"]: x for x in cfg["hypotheses"]}
assert set(specs) == {"H1", "H2", "H3", "H4", "H5"}

mismatches = []
summary_rows = []

for hyp in ["H1", "H2", "H3", "H4", "H5"]:
    group = asr[asr["hypothesis_id"] == hyp]
    spec = specs[hyp]

    assert len(group) == 400
    assert group["audio_id"].nunique() == 400

    before = len(mismatches)

    for row in group.itertuples(index=False):
        expected_beams = spec.get("num_beams")

        if expected_beams is None:
            expected_beams = cfg[
                "runtime_generation_rules"
            ]["num_beams_sampling"]

        observed = {
            "method": str(row.method).strip(),
            "model_id": str(row.model_id).strip(),
            "model_revision": str(row.model_revision).strip(),
            "do_sample": parse_bool(row.do_sample),
            "num_beams": parse_number(row.num_beams),
            "temperature": parse_number(row.temperature),
            "seed": parse_number(row.seed)
        }

        wanted = {
            "method": spec["method"],
            "model_id": cfg["model_id"],
            "model_revision": cfg["model_revision"],
            "do_sample": spec["do_sample"],
            "num_beams": expected_beams,
            "temperature": spec.get("temperature"),
            "seed": spec.get("seed")
        }

        for field in wanted:
            if not compare(observed[field], wanted[field]):
                mismatches.append({
                    "audio_id": row.audio_id,
                    "hypothesis_id": hyp,
                    "field": field,
                    "observed": repr(observed[field]),
                    "expected": repr(wanted[field])
                })

    summary_rows.append({
        "hypothesis_id": hyp,
        "rows": len(group),
        "metadata_mismatches": len(mismatches) - before
    })

mismatch_df = pd.DataFrame(
    mismatches,
    columns=[
        "audio_id", "hypothesis_id", "field",
        "observed", "expected"
    ]
)

summary_df = pd.DataFrame(summary_rows)

# Preserve earlier failed audit files; save new filenames.
def save_csv(frame, path):
    temp = path.with_suffix(".tmp")
    frame.to_csv(temp, index=False, encoding="utf-8")
    if path.exists():
        assert sha(path) == sha(temp), (
            f"Existing corrected file differs: {path}"
        )
        temp.unlink()
    else:
        temp.replace(path)

save_csv(mismatch_df, MISMATCH)
save_csv(summary_df, SUMMARY)

status = (
    "PASS"
    if len(mismatches) == 0
    else "NEEDS_REVIEW"
)

report = {
    "step": "8.4_corrected",
    "status": status,
    "correction_reason": (
        "Prior comparison incorrectly treated Python bool "
        "as a numeric type because bool subclasses int."
    ),
    "previous_report_preserved": True,
    "selected_speakers": 20,
    "selected_audios": 400,
    "asr_hypotheses": 2000,
    "checked_fields_per_hypothesis": 7,
    "metadata_mismatch_count": len(mismatches),
    "sampling_seed_recorded": 42,
    "speaker_selection_algorithm_verified": False,
    "full_inference_source_code_verified": False,
    "max_new_tokens_execution_independently_verified": False,
    "ASR_inference_executed": False,
    "model_training_executed": False,
    "source_sha256": {
        str(path.relative_to(P)): sha(path)
        for path in [ASR, CFG, FREEZE, SPEAKERS, SELECTION]
    },
    "summary_sha256": sha(SUMMARY),
    "mismatch_sha256": sha(MISMATCH)
}

temp = REPORT.with_suffix(".tmp")
temp.write_text(
    json.dumps(report, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

if REPORT.exists():
    assert json.loads(
        REPORT.read_text(encoding="utf-8")
    ) == report
    temp.unlink()
else:
    temp.replace(REPORT)

print("=== STEP 8.4 CORRECTED METADATA AUDIT ===")
print("Status:", status)
print("Total ASR hypotheses:", len(asr))
print("Metadata mismatches:", len(mismatches))
print("Sampling speaker/audio mapping: PASS")
print("Sampling seed recorded:", speaker_cfg["seed"])

print("\n=== PER-HYPOTHESIS RESULT ===")
print(summary_df.to_string(index=False))

print("\n=== MISMATCHES BY FIELD ===")
if mismatch_df.empty:
    print("None")
else:
    print(
        mismatch_df.groupby("field")
        .size()
        .to_string()
    )
    print(mismatch_df.head(20).to_string(index=False))

print("\n=== PROVENANCE STATUS ===")
print("D07 sampling algorithm: NOT YET VERIFIED")
print("D08 saved decoding metadata:",
      "PASS" if status == "PASS" else "NEEDS REVIEW")
print("D08 original inference implementation: NOT YET VERIFIED")

print("\nCorrected report:", REPORT)
print("Report SHA256:", sha(REPORT))
print("Original failed report preserved: YES")


In [ ]:

from pathlib import Path
import json
import re
import hashlib
import pandas as pd

P = Path("/kaggle/working/acoustic_linguistic_research")
W = Path("/kaggle/working")
OUT = P / "results/step08_provenance"
OUT.mkdir(parents=True, exist_ok=True)

SPEAKERS = P / "checkpoints/pilot_speaker_ids.json"
PROGRESS = P / "checkpoints/research_progress.json"
PILOT_CONFIG = P / "configs/pilot_acoustic_v1.json"
SELECTION = P / "data/metadata/pilot_selection_400.csv"

def sha(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

for path in [SPEAKERS, PROGRESS, PILOT_CONFIG, SELECTION]:
    assert path.is_file(), f"Missing: {path}"

speaker_info = json.loads(SPEAKERS.read_text(encoding="utf-8"))
progress = json.loads(PROGRESS.read_text(encoding="utf-8"))
pilot_config = json.loads(PILOT_CONFIG.read_text(encoding="utf-8"))

sel = pd.read_csv(
    SELECTION,
    dtype={"audio_id": "string", "speaker": "string"}
)

assert len(sel) == 400
assert sel["speaker"].nunique() == 20
assert sel["audio_id"].nunique() == 400
assert speaker_info["seed"] == 42
assert set(speaker_info["speaker_ids"]) == set(sel["speaker"])
assert sel.groupby("speaker")["audio_id"].nunique().eq(20).all()

print("=== STEP 8.5 SAMPLING PROVENANCE AUDIT ===")

print("\n=== PILOT ACOUSTIC CONFIG ===")
print(json.dumps(pilot_config, indent=2, ensure_ascii=False))

print("\n=== SAMPLING-RELATED PROGRESS KEYS ===")
pattern = re.compile(
    r"seed|random|sample|select|speaker|pilot|row|index",
    re.IGNORECASE
)

for key, value in progress.items():
    if pattern.search(key):
        if isinstance(value, (dict, list)):
            value = json.dumps(value, ensure_ascii=False)[:500]
        print(f"{key}: {value}")

print("\n=== SELECTION ROW INDEX AUDIT ===")

assert "row" in sel.columns
assert "pilot_index" in sel.columns

sel["row_numeric"] = pd.to_numeric(
    sel["row"], errors="raise"
)
sel["pilot_index_numeric"] = pd.to_numeric(
    sel["pilot_index"], errors="raise"
)

print("Original row min:", sel["row_numeric"].min())
print("Original row max:", sel["row_numeric"].max())
print("Unique original rows:", sel["row_numeric"].nunique())

print(
    "Pilot index consecutive 1-400:",
    sorted(sel["pilot_index_numeric"].tolist())
    == list(range(1, 401))
)

print(
    "Original dataset rows strictly increasing:",
    bool(sel["row_numeric"].is_monotonic_increasing)
)

print(
    "Speaker IDs sorted in selection:",
    sel["speaker"].drop_duplicates().tolist()
    == sorted(sel["speaker"].unique().tolist())
)

print("\n=== SPEAKER-LEVEL ORIGINAL ROW SUMMARY ===")
speaker_rows = (
    sel.groupby("speaker")
    .agg(
        selected_audios=("audio_id", "count"),
        original_row_min=("row_numeric", "min"),
        original_row_max=("row_numeric", "max"),
        selected_index_min=("pilot_index_numeric", "min"),
        selected_index_max=("pilot_index_numeric", "max")
    )
    .reset_index()
)

print(speaker_rows.to_string(index=False))

print("\n=== FIRST AND LAST 5 SELECTIONS ===")
print(
    sel[
        ["pilot_index", "row", "speaker", "audio_id"]
    ].head(5).to_string(index=False)
)
print("...")
print(
    sel[
        ["pilot_index", "row", "speaker", "audio_id"]
    ].tail(5).to_string(index=False)
)

# Inspect only source code, not notebook outputs or unrelated files.
source_files = [
    path for path in P.rglob("*.py")
    if path.is_file()
]

source_files += [
    path for path in W.glob("*.py")
    if path.is_file()
]

source_files += [
    path for path in W.glob("*.ipynb")
    if path.is_file()
]

source_files = sorted(set(source_files))

code_pattern = re.compile(
    r"default_rng|RandomState|random\.sample|"
    r"random\.seed|np\.random|rng\.choice|"
    r"random\.choice|sample\(n=|"
    r"sample\(frac=|speaker_ids|pilot_selection_400",
    re.IGNORECASE
)

matches = []

for path in source_files:
    if path.stat().st_size > 20_000_000:
        continue

    try:
        if path.suffix == ".ipynb":
            obj = json.loads(path.read_text(encoding="utf-8"))
            sections = [
                (
                    f"cell_{i}",
                    "".join(c.get("source", []))
                    if isinstance(c.get("source", []), list)
                    else str(c.get("source", ""))
                )
                for i, c in enumerate(obj.get("cells", []))
                if c.get("cell_type") == "code"
            ]
        else:
            sections = [
                ("source", path.read_text(encoding="utf-8"))
            ]
    except Exception:
        continue

    for section, source in sections:
        for line_no, line in enumerate(source.splitlines(), 1):
            if code_pattern.search(line):
                matches.append({
                    "file": str(path),
                    "section": section,
                    "line": line_no,
                    "snippet": line.strip()[:400],
                    "source_sha256": sha(path)
                })

code_evidence = pd.DataFrame(
    matches,
    columns=[
        "file", "section", "line",
        "snippet", "source_sha256"
    ]
)

CODE_OUT = OUT / "step08_05_sampling_code_evidence.csv"
SPEAKER_OUT = OUT / "step08_05_sampling_speaker_rows.csv"
REPORT = OUT / "step08_05_sampling_provenance.json"

def safe_csv(frame, path):
    temp = path.with_suffix(".tmp")
    frame.to_csv(temp, index=False, encoding="utf-8")
    if path.exists():
        assert sha(path) == sha(temp), (
            f"Existing file differs: {path}"
        )
        temp.unlink()
    else:
        temp.replace(path)

safe_csv(code_evidence, CODE_OUT)
safe_csv(speaker_rows, SPEAKER_OUT)

print("\n=== ORIGINAL SAMPLING CODE EVIDENCE ===")
print("Source files searched:", len(source_files))
print("Matching code lines:", len(code_evidence))

if code_evidence.empty:
    print("No matching source code found.")
else:
    print(
        code_evidence[
            ["file", "section", "line", "snippet"]
        ].head(50).to_string(
            index=False, max_colwidth=110
        )
    )

report = {
    "step": "8.5",
    "status": "EVIDENCE_REVIEW",
    "sampling_seed_documented": 42,
    "speaker_ids_documented": 20,
    "selected_audios_verified": 400,
    "selected_audios_per_speaker": 20,
    "saved_selection_consistent": True,
    "original_sampling_algorithm_verified": False,
    "sampling_source_files_checked": len(source_files),
    "sampling_code_matching_lines": len(code_evidence),
    "source_hashes": {
        str(path.relative_to(P)): sha(path)
        for path in [SPEAKERS, PROGRESS, PILOT_CONFIG, SELECTION]
    },
    "output_hashes": {
        CODE_OUT.name: sha(CODE_OUT),
        SPEAKER_OUT.name: sha(SPEAKER_OUT)
    },
    "note": (
        "Recorded seed and selected rows do not independently "
        "establish the original random sampling algorithm. "
        "Potential code matches require human review."
    )
}

if REPORT.exists():
    assert json.loads(
        REPORT.read_text(encoding="utf-8")
    ) == report
else:
    temp = REPORT.with_suffix(".tmp")
    temp.write_text(
        json.dumps(report, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    temp.replace(REPORT)

print("\n=== STEP 8.5 SUMMARY ===")
print("Status:", report["status"])
print("Sampling seed:", report["sampling_seed_documented"])
print("Selected speaker/audio mapping: PASS")
print("Original algorithm independently verified: NO")
print("Code evidence CSV:", CODE_OUT)
print("Report:", REPORT)
print("Report SHA256:", sha(REPORT))
print("No ASR inference or model training performed.")


In [ ]:

from pathlib import Path
import json
import hashlib
import random

import numpy as np
import pandas as pd

P = Path("/kaggle/working/acoustic_linguistic_research")
OUT = P / "results/step08_provenance"
OUT.mkdir(parents=True, exist_ok=True)

SELECTION = P / "data/metadata/pilot_selection_400.csv"
SPEAKERS = P / "checkpoints/pilot_speaker_ids.json"

REPORT = OUT / "step08_06_sampling_reconstruction.json"
RESULTS = OUT / "step08_06_sampling_candidate_comparison.csv"

def sha(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

sel = pd.read_csv(
    SELECTION,
    dtype={"audio_id": "string", "speaker": "string"}
)
saved = json.loads(SPEAKERS.read_text(encoding="utf-8"))

assert saved["seed"] == 42
assert len(sel) == 400
assert len(saved["speaker_ids"]) == 20

sel["row"] = pd.to_numeric(sel["row"], errors="raise")

speaker_rows = (
    sel.groupby("speaker")
    .agg(
        row_min=("row", "min"),
        row_max=("row", "max"),
        n=("row", "nunique")
    )
    .reset_index()
    .sort_values("row_min")
)

assert len(speaker_rows) == 20
assert speaker_rows["n"].eq(20).all()

# Check all 20 consecutive original rows are present.
contiguous = True

for speaker, group in sel.groupby("speaker"):
    rows = sorted(group["row"].astype(int).tolist())
    if rows != list(range(rows[0], rows[0] + 20)):
        contiguous = False

assert contiguous

# Hypothesis: 125 speakers occupy contiguous 20-row
# blocks in the full train dataset, with 1-based row IDs.
block_aligned = bool(
    ((speaker_rows["row_min"] - 1) % 20 == 0).all()
)

assert block_aligned

observed_zero_based = sorted(
    ((speaker_rows["row_min"] - 1) // 20)
    .astype(int)
    .tolist()
)

assert len(set(observed_zero_based)) == 20
assert min(observed_zero_based) >= 0
assert max(observed_zero_based) < 125

candidate_methods = {
    "numpy_default_rng_choice":
        np.random.default_rng(42).choice(
            125, size=20, replace=False
        ).tolist(),

    "numpy_RandomState_choice":
        np.random.RandomState(42).choice(
            125, size=20, replace=False
        ).tolist(),

    "python_random_sample":
        random.Random(42).sample(
            range(125), 20
        ),

    "numpy_default_rng_permutation":
        np.random.default_rng(42).permutation(125)[:20].tolist(),

    "numpy_RandomState_permutation":
        np.random.RandomState(42).permutation(125)[:20].tolist()
}

records = []

for method, selected in candidate_methods.items():
    selected_sorted = sorted(int(x) for x in selected)

    overlap = len(
        set(selected_sorted) & set(observed_zero_based)
    )

    exact_set_match = (
        selected_sorted == observed_zero_based
    )

    records.append({
        "method": method,
        "seed": 42,
        "speaker_population_assumed": 125,
        "sample_size": 20,
        "matching_positions": overlap,
        "exact_set_match": exact_set_match,
        "candidate_positions_zero_based": ",".join(
            map(str, selected_sorted)
        )
    })

results = pd.DataFrame(records)

def safe_csv(frame, path):
    temp = path.with_suffix(".tmp")
    frame.to_csv(temp, index=False, encoding="utf-8")
    if path.exists():
        assert sha(path) == sha(temp), (
            f"Existing result differs: {path}"
        )
        temp.unlink()
    else:
        temp.replace(path)

safe_csv(results, RESULTS)

matching = results.loc[
    results["exact_set_match"],
    "method"
].tolist()

report = {
    "step": "8.6",
    "status": "RECONSTRUCTION_COMPLETED",
    "recorded_seed": 42,
    "selected_speakers": 20,
    "selected_audios": 400,
    "audio_rows_consecutive_within_speaker": contiguous,
    "speaker_row_blocks_aligned_to_20": block_aligned,
    "observed_speaker_positions_zero_based": observed_zero_based,
    "candidate_algorithms_tested": len(candidate_methods),
    "exact_matching_candidates": matching,
    "original_algorithm_verified": False,
    "critical_assumption": (
        "All 125 speakers in the original train partition "
        "occupy 20-row blocks in the same order, and "
        "the original selection sampled zero-based positions."
    ),
    "limitations": [
        "Full 125-speaker source ordering was not independently loaded.",
        "An exact random sample match does not prove original code identity.",
        "A mismatch does not exclude other legitimate selection algorithms.",
        "This is a reconstruction audit, not an original code audit."
    ],
    "source_sha256": {
        str(SELECTION.relative_to(P)): sha(SELECTION),
        str(SPEAKERS.relative_to(P)): sha(SPEAKERS)
    },
    "results_sha256": sha(RESULTS),
    "model_retraining": False,
    "asr_inference": False
}

if REPORT.exists():
    assert json.loads(
        REPORT.read_text(encoding="utf-8")
    ) == report
else:
    temp = REPORT.with_suffix(".tmp")
    temp.write_text(
        json.dumps(report, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    temp.replace(REPORT)

print("=== STEP 8.6 SAMPLING RECONSTRUCTION ===")
print("Status: RECONSTRUCTION_COMPLETED")
print("Recorded seed:", saved["seed"])
print("Speakers:", len(speaker_rows))
print("Audios:", len(sel))
print("Contiguous audio blocks:", contiguous)
print("20-row block alignment:", block_aligned)

print("\n=== OBSERVED SPEAKER POSITIONS ===")
print("Zero-based:", observed_zero_based)

print("\n=== CANDIDATE ALGORITHM COMPARISON ===")
print(
    results[
        [
            "method",
            "matching_positions",
            "exact_set_match"
        ]
    ].to_string(index=False)
)

print("\n=== EXACT MATCHING CANDIDATES ===")
print(matching if matching else "None")

print("\n=== INTERPRETATION BOUNDARY ===")
print("Original source algorithm proven: NO")
print("125-speaker original order verified: NO")
print("Sampling reconstruction is exploratory.")

print("\nReport:", REPORT)
print("Report SHA256:", sha(REPORT))
print("Comparison CSV:", RESULTS)
print("No model training or ASR inference performed.")


In [ ]:

from pathlib import Path
import json
import hashlib
import numpy as np
import pandas as pd

P = Path("/kaggle/working/acoustic_linguistic_research")
OUT = P / "results/step08_provenance"
OUT.mkdir(parents=True, exist_ok=True)

RAW = P / "data/raw/speechocean762_train.parquet"
SELECTION = P / "data/metadata/pilot_selection_400.csv"
SPEAKERS = P / "checkpoints/pilot_speaker_ids.json"

REPORT = OUT / "step08_07_source_sampling_reconstruction.json"
COMPARE = OUT / "step08_07_reconstructed_selection.csv"

def sha(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

assert RAW.is_file(), (
    "Original train parquet is unavailable. "
    "Reattach the pinned dataset before this audit."
)

speaker_info = json.loads(SPEAKERS.read_text(encoding="utf-8"))
selection = pd.read_csv(
    SELECTION,
    dtype={"audio_id": "string", "speaker": "string"}
)

# Read metadata columns only; no audio bytes decoded.
import pyarrow.parquet as pq

schema = pq.read_schema(RAW)
available = set(schema.names)

assert "audio_id" in available
assert "speaker" in available

raw = pq.read_table(
    RAW,
    columns=["audio_id", "speaker"]
).to_pandas()

raw["audio_id"] = raw["audio_id"].astype("string")
raw["speaker"] = raw["speaker"].astype("string")

assert len(raw) == 2500
assert raw["speaker"].nunique() == 125
assert raw["audio_id"].nunique() == 2500

raw["original_row_1based"] = np.arange(1, 2501)

# Speaker order in the actual stored train data.
original_speaker_order = (
    raw["speaker"].drop_duplicates().tolist()
)

speaker_counts = raw["speaker"].value_counts()
all_have_20 = bool(speaker_counts.eq(20).all())

contiguous_blocks = True

for speaker, g in raw.groupby("speaker", sort=False):
    positions = g["original_row_1based"].to_numpy()
    if not np.array_equal(
        positions,
        np.arange(positions[0], positions[0] + len(positions))
    ):
        contiguous_blocks = False
        break

seed = int(speaker_info["seed"])
rng = np.random.default_rng(seed)

selected_positions = rng.choice(
    len(original_speaker_order),
    size=20,
    replace=False
)

reconstructed_speakers = sorted(
    original_speaker_order[int(i)]
    for i in selected_positions
)

saved_speakers = sorted(speaker_info["speaker_ids"])

speaker_match = reconstructed_speakers == saved_speakers

# Reconstruct: selected speakers, all their 20 source
# rows in original source order, speakers sorted.
reconstructed = (
    raw[raw["speaker"].isin(reconstructed_speakers)]
    .sort_values(
        ["speaker", "original_row_1based"]
    )
    .reset_index(drop=True)
)

reconstructed.insert(
    0, "pilot_index", np.arange(1, len(reconstructed) + 1)
)

assert len(reconstructed) == 400

selected = selection[
    ["pilot_index", "row", "audio_id", "speaker"]
].copy()

selected["pilot_index"] = pd.to_numeric(
    selected["pilot_index"], errors="raise"
).astype(int)

selected["row"] = pd.to_numeric(
    selected["row"], errors="raise"
).astype(int)

selected = selected.sort_values(
    "pilot_index"
).reset_index(drop=True)

audio_match = (
    reconstructed["audio_id"].tolist()
    == selected["audio_id"].tolist()
)

row_match = (
    reconstructed["original_row_1based"].tolist()
    == selected["row"].tolist()
)

order_match = (
    reconstructed["pilot_index"].tolist()
    == selected["pilot_index"].tolist()
)

reconstructed["saved_audio_id"] = selected["audio_id"]
reconstructed["saved_row"] = selected["row"]
reconstructed["audio_id_match"] = (
    reconstructed["audio_id"] == reconstructed["saved_audio_id"]
)
reconstructed["row_match"] = (
    reconstructed["original_row_1based"]
    == reconstructed["saved_row"]
)

# Preserve output without silently replacing previous evidence.
tmp = COMPARE.with_suffix(".tmp")
reconstructed.to_csv(
    tmp, index=False, encoding="utf-8"
)
if COMPARE.exists():
    assert sha(COMPARE) == sha(tmp)
    tmp.unlink()
else:
    tmp.replace(COMPARE)

full_match = (
    all_have_20
    and contiguous_blocks
    and speaker_match
    and audio_match
    and row_match
    and order_match
)

report = {
    "step": "8.7",
    "status": (
        "RECONSTRUCTION_MATCH"
        if full_match else "NEEDS_REVIEW"
    ),
    "source_train_rows": len(raw),
    "source_speakers": len(original_speaker_order),
    "each_source_speaker_has_20_rows": all_have_20,
    "source_speaker_blocks_contiguous": contiguous_blocks,
    "recorded_seed": seed,
    "selected_speaker_ids_match": speaker_match,
    "selected_audio_ids_match": audio_match,
    "selected_original_rows_match": row_match,
    "pilot_index_order_match": order_match,
    "original_source_code_verified": False,
    "selection_rule_reconstructed": (
        "NumPy default_rng(seed).choice(125,20,replace=False), "
        "followed by original source row selection "
        "for selected speakers and speaker-sorted output."
    ),
    "parquet_schema_columns_used": ["audio_id", "speaker"],
    "raw_audio_decoded": False,
    "model_training": False,
    "ASR_inference": False,
    "source_hashes": {
        "original_train_parquet": sha(RAW),
        "pilot_selection_400.csv": sha(SELECTION),
        "pilot_speaker_ids.json": sha(SPEAKERS)
    },
    "reconstructed_selection_sha256": sha(COMPARE),
    "limitation": (
        "Exact reconstruction provides strong evidence for "
        "the sampling procedure but does not prove the "
        "historical source code used this exact implementation."
    )
}

if REPORT.exists():
    assert json.loads(
        REPORT.read_text(encoding="utf-8")
    ) == report
else:
    tmp = REPORT.with_suffix(".tmp")
    tmp.write_text(
        json.dumps(report, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    tmp.replace(REPORT)

print("=== STEP 8.7 ORIGINAL DATASET RECONSTRUCTION ===")
print("Status:", report["status"])
print("Original training rows:", len(raw))
print("Original speakers:", len(original_speaker_order))
print("Every speaker has 20 rows:", all_have_20)
print("Speaker rows contiguous:", contiguous_blocks)

print("\n=== RECONSTRUCTION RESULTS ===")
print("Speaker IDs match:", speaker_match)
print("Audio IDs match:", audio_match)
print("Original row indices match:", row_match)
print("Pilot index order match:", order_match)
print("Full reconstruction match:", full_match)

print("\n=== RECONSTRUCTED SPEAKERS ===")
print(reconstructed_speakers)

print("\n=== RECONSTRUCTED SAMPLE PREVIEW ===")
print(
    reconstructed[
        ["pilot_index", "original_row_1based",
         "audio_id", "speaker", "audio_id_match", "row_match"]
    ].head(12).to_string(index=False)
)

print("\n=== METHODOLOGICAL BOUNDARY ===")
print("Original source code recovered: NO")
print("Reconstruction independently matched:", full_match)
print("Source audio decoded: NO")

print("\nReport:", REPORT)
print("Report SHA256:", sha(REPORT))
print("Comparison CSV:", COMPARE)


In [ ]:

from pathlib import Path
import json
import pyarrow.parquet as pq
import pandas as pd

P = Path("/kaggle/working/acoustic_linguistic_research")

RAW = P / "data/raw/speechocean762_train.parquet"
SEL = P / "data/metadata/pilot_selection_400.csv"

assert RAW.is_file(), f"Missing: {RAW}"
assert SEL.is_file(), f"Missing: {SEL}"

pf = pq.ParquetFile(RAW)
schema = pf.schema_arrow

print("=== STEP 8.7A ORIGINAL PARQUET SCHEMA ===")
print("Parquet rows:", pf.metadata.num_rows)
print("Row groups:", pf.metadata.num_row_groups)

print("\n=== ALL TOP-LEVEL COLUMNS ===")
for field in schema:
    print(f"{field.name}: {field.type}")

print("\n=== SELECTION METADATA COLUMNS ===")
sel = pd.read_csv(
    SEL,
    nrows=5,
    dtype={"audio_id": "string", "speaker": "string"}
)

print(sel.columns.tolist())

print("\n=== FIRST 5 SAVED SELECTION ROWS ===")
available_selection = [
    c for c in [
        "pilot_index", "row", "speaker",
        "audio_id", "audio_path", "split"
    ]
    if c in sel.columns
]

print(
    sel[available_selection].to_string(index=False)
)

print("\n=== POSSIBLE SOURCE METADATA COLUMNS ===")
metadata_cols = [
    name for name in schema.names
    if any(token in name.lower() for token in [
        "audio", "path", "file", "speaker",
        "id", "row", "text", "fluency"
    ])
    and name.lower() != "audio"
]

print(metadata_cols)

# Read a tiny number of rows from metadata columns only.
# Do not access the potentially large audio column.
safe_cols = [
    name for name in metadata_cols
    if not (
        name.lower() == "audio"
        or "binary" in str(schema.field(name).type).lower()
    )
]

safe_cols = safe_cols[:10]

if safe_cols:
    batch = next(
        pf.iter_batches(
            batch_size=5,
            columns=safe_cols
        ),
        None
    )

    if batch is not None:
        example = batch.to_pandas()

        print("\n=== FIRST 5 SOURCE METADATA ROWS ===")
        print(
            example.to_string(
                index=False,
                max_colwidth=100
            )
        )
else:
    print("\nNo suitable metadata columns found.")

print("\n=== DIAGNOSTIC SUMMARY ===")
print("Schema inspection: COMPLETE")
print("Audio decoded: NO")
print("ASR inference: NO")
print("Model training: NO")
print("Original files modified: NO")
print("Next: adapt STEP 8.7 to actual source schema")


In [ ]:

from pathlib import Path
import json
import hashlib

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

P = Path("/kaggle/working/acoustic_linguistic_research")
OUT = P / "results/step08_provenance"
OUT.mkdir(parents=True, exist_ok=True)

RAW = P / "data/raw/speechocean762_train.parquet"
SEL = P / "data/metadata/pilot_selection_400.csv"
SPEAKERS = P / "checkpoints/pilot_speaker_ids.json"

REPORT = OUT / "step08_07b_source_reconstruction.json"
COMPARE = OUT / "step08_07b_reconstructed_selection.csv"

def sha(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

for path in [RAW, SEL, SPEAKERS]:
    assert path.is_file(), f"Missing: {path}"

saved = json.loads(SPEAKERS.read_text(encoding="utf-8"))

selection = pd.read_csv(
    SEL,
    dtype={"speaker": "string", "audio_id": "string"}
)

selection["row"] = pd.to_numeric(
    selection["row"], errors="raise"
).astype(int)

selection["pilot_index"] = pd.to_numeric(
    selection["pilot_index"], errors="raise"
).astype(int)

selection = selection.sort_values(
    "pilot_index"
).reset_index(drop=True)

# Read metadata only, excluding audio.bytes.
pf = pq.ParquetFile(RAW)

leaf_paths = [
    pf.schema.column(i).path
    for i in range(len(pf.schema.names))
]

assert "audio.path" in leaf_paths, leaf_paths

table = pf.read(
    columns=["speaker", "audio.path", "fluency", "text"]
)

raw = table.to_pandas()

# Arrow may expose the nested field as a struct
# rather than a flattened column.
if "audio.path" in raw.columns:
    paths = raw["audio.path"]
elif "audio" in raw.columns:
    paths = raw["audio"].map(
        lambda x: x.get("path")
        if isinstance(x, dict) else None
    )
else:
    raise AssertionError(
        f"Audio path missing from projection: {list(raw.columns)}"
    )

raw["speaker"] = raw["speaker"].astype("string")
raw["audio_path_source"] = paths.astype("string")

assert raw["audio_path_source"].notna().all()
assert len(raw) == 2500
assert raw["speaker"].nunique() == 125

raw["audio_id"] = raw["audio_path_source"].map(
    lambda x: Path(str(x)).stem
).astype("string")

raw["original_row_1based"] = np.arange(1, 2501)

assert raw["audio_id"].nunique() == 2500

original_speaker_order = (
    raw["speaker"].drop_duplicates().tolist()
)

counts = raw.groupby("speaker")["audio_id"].nunique()
all_have_20 = bool(counts.eq(20).all())

contiguous = all(
    np.array_equal(
        group["original_row_1based"].to_numpy(),
        np.arange(
            group["original_row_1based"].iloc[0],
            group["original_row_1based"].iloc[0] + len(group)
        )
    )
    for _, group in raw.groupby("speaker", sort=False)
)

seed = int(saved["seed"])
positions = np.random.default_rng(seed).choice(
    len(original_speaker_order),
    size=20,
    replace=False
)

chosen_speakers = sorted(
    original_speaker_order[int(i)]
    for i in positions
)

speaker_match = (
    chosen_speakers == sorted(saved["speaker_ids"])
)

reconstructed = (
    raw[raw["speaker"].isin(chosen_speakers)]
    .sort_values(["speaker", "original_row_1based"])
    .reset_index(drop=True)
)

reconstructed.insert(
    0, "pilot_index", np.arange(1, len(reconstructed) + 1)
)

assert len(reconstructed) == 400

reconstructed["saved_audio_id"] = selection["audio_id"]
reconstructed["saved_speaker"] = selection["speaker"]
reconstructed["saved_row"] = selection["row"]

reconstructed["audio_id_match"] = (
    reconstructed["audio_id"]
    == reconstructed["saved_audio_id"]
)

reconstructed["speaker_match"] = (
    reconstructed["speaker"]
    == reconstructed["saved_speaker"]
)

reconstructed["row_match"] = (
    reconstructed["original_row_1based"]
    == reconstructed["saved_row"]
)

index_match = (
    reconstructed["pilot_index"].tolist()
    == selection["pilot_index"].tolist()
)

audio_match = bool(
    reconstructed["audio_id_match"].all()
)
row_match = bool(
    reconstructed["row_match"].all()
)
speaker_row_match = bool(
    reconstructed["speaker_match"].all()
)

# Also compare labels, prompt text and source path.
fluency_match = bool(np.allclose(
    reconstructed["fluency"].astype(float),
    selection["fluency"].astype(float)
))

text_match = bool(
    reconstructed["text"].astype("string")
    .fillna("<missing>")
    .eq(selection["text"].astype("string").fillna("<missing>"))
    .all()
)

path_match = bool(
    reconstructed["audio_path_source"]
    .map(lambda x: Path(str(x)).name)
    .eq(
        selection["audio_path"].astype("string")
        .map(lambda x: Path(str(x)).name)
    )
    .all()
)

full_match = all([
    all_have_20,
    contiguous,
    speaker_match,
    speaker_row_match,
    audio_match,
    row_match,
    index_match,
    fluency_match,
    text_match,
    path_match
])

def safe_csv(frame, path):
    tmp = path.with_suffix(".tmp")
    frame.to_csv(
        tmp, index=False, encoding="utf-8"
    )
    if path.exists():
        assert sha(path) == sha(tmp), (
            f"Existing output differs: {path}"
        )
        tmp.unlink()
    else:
        tmp.replace(path)

# No raw audio bytes are included in this CSV.
save_columns = [
    "pilot_index", "original_row_1based",
    "speaker", "audio_id", "audio_path_source",
    "saved_speaker", "saved_audio_id", "saved_row",
    "speaker_match", "audio_id_match", "row_match"
]

safe_csv(reconstructed[save_columns], COMPARE)

report = {
    "step": "8.7b",
    "status": (
        "RECONSTRUCTION_MATCH"
        if full_match else "NEEDS_REVIEW"
    ),
    "source_rows": len(raw),
    "source_speakers": len(original_speaker_order),
    "sampling_seed": seed,
    "candidate_algorithm":
        "numpy.default_rng(42).choice(125,20,replace=False)",
    "all_source_speakers_have_20_rows": all_have_20,
    "speaker_blocks_contiguous": contiguous,
    "selected_speakers_match": speaker_match,
    "speaker_mapping_match": speaker_row_match,
    "selected_audio_ids_match": audio_match,
    "source_row_indices_match": row_match,
    "pilot_index_match": index_match,
    "fluency_labels_match": fluency_match,
    "reading_prompts_match": text_match,
    "audio_path_basename_match": path_match,
    "original_selection_source_code_found": False,
    "audio_bytes_read": False,
    "asr_inference": False,
    "model_training": False,
    "source_sha256": {
        str(path.relative_to(P)): sha(path)
        for path in [RAW, SEL, SPEAKERS]
    },
    "comparison_sha256": sha(COMPARE),
    "interpretation": (
        "An exact match supports reconstruction of the "
        "original sample from the pinned training data. "
        "It does not prove the historical code identity."
    )
}

tmp = REPORT.with_suffix(".tmp")
tmp.write_text(
    json.dumps(report, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

if REPORT.exists():
    assert json.loads(
        REPORT.read_text(encoding="utf-8")
    ) == report, "Existing report differs"
    tmp.unlink()
else:
    tmp.replace(REPORT)

print("=== STEP 8.7B SOURCE SAMPLING RECONSTRUCTION ===")
print("Status:", report["status"])
print("Original train rows:", len(raw))
print("Original train speakers:", len(original_speaker_order))
print("All speakers have 20 rows:", all_have_20)
print("Speaker blocks contiguous:", contiguous)

print("\n=== RECONSTRUCTION RESULTS ===")
for key in [
    "selected_speakers_match",
    "speaker_mapping_match",
    "selected_audio_ids_match",
    "source_row_indices_match",
    "pilot_index_match",
    "fluency_labels_match",
    "reading_prompts_match",
    "audio_path_basename_match"
]:
    print(key + ":", report[key])

print("FULL MATCH:", full_match)

print("\n=== RECONSTRUCTED SPEAKERS ===")
print(chosen_speakers)

print("\n=== FIRST 5 RECONSTRUCTED ROWS ===")
print(
    reconstructed[save_columns]
    .head(5)
    .to_string(index=False)
)

print("\n=== LIMITATION ===")
print("Historical sampling source code recovered: NO")
print("Exact source-based reconstruction:", full_match)

print("\nReport:", REPORT)
print("Report SHA256:", sha(REPORT))
print("Comparison CSV:", COMPARE)


In [ ]:

from pathlib import Path
import json
import hashlib
import numpy as np
import pandas as pd

P = Path("/kaggle/working/acoustic_linguistic_research")
OUT = P / "results/step08_provenance"
OUT.mkdir(parents=True, exist_ok=True)

BASE = P / "features/asr/full"
FINAL = BASE / "asr_hypotheses_0001-0400.csv"
CONFIG = P / "configs/asr_hypothesis_pilot_v1.json"
PROGRESS = P / "checkpoints/research_progress.json"

BATCHES = {
    "0001-0100": BASE / "asr_hypotheses_0001-0100.csv",
    "0101-0200": BASE / "asr_hypotheses_0101-0200.csv",
    "0201-0300": BASE / "asr_hypotheses_0201-0300.csv",
    "0301-0400": BASE / "asr_hypotheses_0301-0400.csv",
}

REPORT = OUT / "step08_08_asr_batch_integrity.json"
BATCH_REPORT = OUT / "step08_08_batch_comparison.csv"
DIFFERENCES = OUT / "step08_08_row_differences.csv"

def sha(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def read(path):
    return pd.read_csv(
        path,
        dtype={
            "audio_id": "string",
            "speaker": "string",
            "hypothesis_id": "string"
        }
    )

assert sha(FINAL) == (
    "b67dfe5cca2329407ab07658809434c2e3bad00d227485f3f8c8c9c1dbbf3e80"
)

final = read(FINAL)
cfg = json.loads(CONFIG.read_text(encoding="utf-8"))
progress = json.loads(PROGRESS.read_text(encoding="utf-8"))

KEYS = [
    "pilot_index", "audio_id",
    "speaker", "hypothesis_id"
]

FIELDS = [
    "method", "model_id", "model_revision",
    "do_sample", "num_beams",
    "temperature", "seed", "transcription",
    "duration_sec", "inference_sec"
]

assert len(final) == 2000
assert not final.duplicated(KEYS).any()
assert set(KEYS + FIELDS).issubset(final.columns)

# Canonical comparison: string columns must agree,
# while floating-point columns tolerate only
# roundtrip serialization differences.
TEXT_FIELDS = [
    "method", "model_id",
    "model_revision", "transcription"
]

NUMBER_FIELDS = [
    "num_beams", "temperature",
    "seed", "duration_sec", "inference_sec"
]

def equal_series(left, right, column):
    if column in TEXT_FIELDS:
        a = left.astype("string")
        b = right.astype("string")
        return (
            a.eq(b).fillna(False)
            | (a.isna() & b.isna())
        ).to_numpy(dtype=bool)

    if column == "do_sample":
        a = left.astype("string").str.lower()
        b = right.astype("string").str.lower()
        return (
            a.eq(b).fillna(False)
            | (a.isna() & b.isna())
        ).to_numpy(dtype=bool)

    a = pd.to_numeric(left, errors="coerce")
    b = pd.to_numeric(right, errors="coerce")

    return np.isclose(
        a.to_numpy(dtype=float),
        b.to_numpy(dtype=float),
        rtol=1e-10,
        atol=1e-10,
        equal_nan=True
    )

batch_frames = []
batch_summary = []

for label, path in BATCHES.items():
    assert path.is_file(), f"Missing batch: {path}"

    part = read(path)

    assert len(part) == 500, (
        f"Unexpected rows in {label}: {len(part)}"
    )
    assert part["audio_id"].nunique() == 100
    assert not part.duplicated(KEYS).any()
    assert set(KEYS + FIELDS).issubset(part.columns)

    # The expected batch row range is based on
    # pilot_index rather than audio ID.
    low, high = map(int, label.split("-"))

    assert part["pilot_index"].between(low, high).all()

    batch_frames.append(part)

    batch_summary.append({
        "batch": label,
        "rows": len(part),
        "unique_audios": part["audio_id"].nunique(),
        "unique_hypotheses": part["hypothesis_id"].nunique(),
        "sha256": sha(path)
    })

combined = pd.concat(batch_frames, ignore_index=True)

assert len(combined) == 2000
assert not combined.duplicated(KEYS).any()

joined = final[KEYS + FIELDS].merge(
    combined[KEYS + FIELDS],
    on=KEYS,
    how="outer",
    validate="one_to_one",
    suffixes=("_final", "_batch"),
    indicator=True
)

missing_keys = int(joined["_merge"].ne("both").sum())

differences = []

for field in FIELDS:
    left = joined[f"{field}_final"]
    right = joined[f"{field}_batch"]

    matches = equal_series(left, right, field)

    for idx in np.flatnonzero(~matches):
        row = joined.iloc[int(idx)]
        differences.append({
            "pilot_index": row["pilot_index"],
            "audio_id": row["audio_id"],
            "hypothesis_id": row["hypothesis_id"],
            "field": field,
            "batch_value": str(row[f"{field}_batch"])[:300],
            "final_value": str(row[f"{field}_final"])[:300]
        })

diff = pd.DataFrame(
    differences,
    columns=[
        "pilot_index", "audio_id", "hypothesis_id",
        "field", "batch_value", "final_value"
    ]
)

batch_table = pd.DataFrame(batch_summary)

# Check the recorded batch hashes, when present.
recorded_hash_checks = {}

for label, path in BATCHES.items():
    key = f"step3_full_batch_{label}_sha256"
    expected_hash = progress.get(key)

    recorded_hash_checks[label] = {
        "progress_hash_available": expected_hash is not None,
        "matches": (
            sha(path) == expected_hash
            if expected_hash is not None else None
        )
    }

def safe_csv(frame, path):
    temp = path.with_suffix(".tmp")
    frame.to_csv(
        temp, index=False, encoding="utf-8"
    )
    if path.exists():
        assert sha(path) == sha(temp), (
            f"Existing file differs: {path}"
        )
        temp.unlink()
    else:
        temp.replace(path)

safe_csv(batch_table, BATCH_REPORT)
safe_csv(diff, DIFFERENCES)

all_recorded_hashes_match = all(
    item["matches"] is True
    for item in recorded_hash_checks.values()
)

status = (
    "PASS"
    if missing_keys == 0
    and len(diff) == 0
    and all_recorded_hashes_match
    else "NEEDS_REVIEW"
)

report = {
    "step": "8.8",
    "status": status,
    "original_batch_count": 4,
    "combined_hypothesis_rows": len(combined),
    "final_hypothesis_rows": len(final),
    "missing_or_extra_keys": missing_keys,
    "field_level_differences": len(diff),
    "progress_batch_hashes": recorded_hash_checks,
    "frozen_asr_model_id": cfg["model_id"],
    "frozen_asr_model_revision": cfg["model_revision"],
    "saved_metadata_audit":
        "STEP 8.4 CORRECTED PASS",
    "historical_inference_code_recovered": False,
    "runtime_generation_rules_independently_verified": False,
    "raw_audio_decoded": False,
    "asr_inference_performed": False,
    "model_training_performed": False,
    "batch_report_sha256": sha(BATCH_REPORT),
    "differences_sha256": sha(DIFFERENCES),
    "source_hashes": {
        str(path.relative_to(P)): sha(path)
        for path in [FINAL, CONFIG, PROGRESS, *BATCHES.values()]
    }
}

temp = REPORT.with_suffix(".tmp")
temp.write_text(
    json.dumps(report, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

if REPORT.exists():
    assert json.loads(
        REPORT.read_text(encoding="utf-8")
    ) == report, "Existing report differs"
    temp.unlink()
else:
    temp.replace(REPORT)

print("=== STEP 8.8 ASR BATCH INTEGRITY AUDIT ===")
print("Status:", status)
print("Final hypothesis rows:", len(final))
print("Combined batch rows:", len(combined))
print("Missing/extra keys:", missing_keys)
print("Field-level differences:", len(diff))

print("\n=== BATCH HASH VERIFICATION ===")
for label, result in recorded_hash_checks.items():
    print(label, result)

print("\n=== BATCH SUMMARY ===")
print(
    batch_table[
        ["batch", "rows", "unique_audios",
         "unique_hypotheses"]
    ].to_string(index=False)
)

print("\n=== DIFFERENCE EXAMPLES ===")
if diff.empty:
    print("None")
else:
    print(diff.head(15).to_string(index=False))

print("\n=== D08 EVIDENCE BOUNDARY ===")
print("Saved generation metadata: VERIFIED")
print("Batch-to-final consistency:",
      "PASS" if status == "PASS" else "NEEDS REVIEW")
print("Historical inference source code: NOT RECOVERED")
print("Full runtime independently replayed: NO")

print("\nReport:", REPORT)
print("Report SHA256:", sha(REPORT))
print("Batch CSV:", BATCH_REPORT)
print("Difference CSV:", DIFFERENCES)


In [ ]:

from pathlib import Path
import json
import hashlib
import pandas as pd

P = Path("/kaggle/working/acoustic_linguistic_research")
OUT = P / "results/step08_provenance"
OUT.mkdir(parents=True, exist_ok=True)

SOURCES = {
    "D07_reconstruction": (
        OUT / "step08_07b_source_reconstruction.json",
        "64f25a9b45a8720ef2b197630fef16b325b193189b69d6b17396efb18eeeea97"
    ),
    "D08_metadata": (
        OUT / "step08_04_corrected_metadata_audit.json",
        "e8e1f3ac5c5ea143e18114230fc05f012b33d2e19f4eb354794b36b0c37cc45f"
    ),
    "D08_batches": (
        OUT / "step08_08_asr_batch_integrity.json",
        "6fbe46b3093d8cd23b817b33f1ec89e07f8b9e08aaf979277bb5b8b33b608c88"
    )
}

RECORD = OUT / "step08_09_final_provenance_record.json"
TABLE = OUT / "step08_09_provenance_decisions.csv"

def sha(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

audits = {}

for name, (path, expected) in SOURCES.items():
    assert path.is_file(), f"Missing source: {path}"
    assert sha(path) == expected, f"SHA mismatch: {name}"
    audits[name] = json.loads(path.read_text(encoding="utf-8"))

d07 = audits["D07_reconstruction"]
d08_metadata = audits["D08_metadata"]
d08_batches = audits["D08_batches"]

assert d07["status"] == "RECONSTRUCTION_MATCH"
assert d07["selected_speakers_match"] is True
assert d07["selected_audio_ids_match"] is True
assert d07["source_row_indices_match"] is True
assert d07["fluency_labels_match"] is True
assert d07["reading_prompts_match"] is True

assert d08_metadata["status"] == "PASS"
assert d08_metadata["metadata_mismatch_count"] == 0
assert d08_batches["status"] == "PASS"
assert d08_batches["missing_or_extra_keys"] == 0
assert d08_batches["field_level_differences"] == 0

assert all(
    x["matches"] is True
    for x in d08_batches["progress_batch_hashes"].values()
)

decisions = [
    {
        "decision_id": "D07",
        "status": "EXACT_SOURCE_BASED_RECONSTRUCTION",
        "verified": (
            "20 speaker IDs, all 400 selected audio IDs, "
            "source rows, prompt texts, fluency labels and "
            "audio paths reproduced from source train parquet."
        ),
        "limitation": (
            "Historical sampling source code unavailable; "
            "exact reconstruction does not establish code identity."
        )
    },
    {
        "decision_id": "D08",
        "status": "SAVED_ASR_ARTIFACT_PROVENANCE_VERIFIED",
        "verified": (
            "Frozen model revision and H1-H5 metadata match "
            "all 2000 saved ASR hypotheses. Four original "
            "batches exactly match final dataset."
        ),
        "limitation": (
            "Historical full-run inference code unavailable; "
            "original GPU inference not independently replayed."
        )
    },
    {
        "decision_id": "D09",
        "status": "LIMITATION_RETAINED",
        "verified": "No ASR pretraining-overlap audit performed.",
        "limitation": (
            "Do not claim absence of overlap between "
            "SpeechOcean762 and ASR pretraining data."
        )
    },
    {
        "decision_id": "D10",
        "status": "SCOPE_RESTRICTION",
        "verified": (
            "Prompt-aware scores are evaluated under "
            "a read-aloud task with available prompts."
        ),
        "limitation": (
            "Actual deployment prompt availability "
            "has not been independently established."
        )
    },
    {
        "decision_id": "D11",
        "status": "LIMITATION_RETAINED",
        "verified": (
            "Speaker-disjoint internal cross-validation "
            "was completed on the selected train subset."
        ),
        "limitation": (
            "No external validation or evaluation "
            "on the sealed official test partition."
        )
    }
]

table = pd.DataFrame(decisions)

methods_sampling = (
    "The analysis used 400 recordings from 20 speakers "
    "(20 recordings per speaker) selected from the "
    "SpeechOcean762 training partition. The saved sample "
    "was exactly reconstructed from the original source "
    "metadata using a NumPy default_rng(42) choice of "
    "20 out of 125 speaker positions without replacement, "
    "followed by selection of all 20 recordings for each "
    "selected speaker. Although the original sampling "
    "source code was unavailable, the reconstructed "
    "speaker IDs, recording IDs, source-row positions, "
    "fluency labels, reading prompts, and audio paths "
    "matched the frozen selection manifest."
)

methods_asr = (
    "Five ASR hypotheses per recording were generated "
    "using a frozen configuration documenting "
    "openai/whisper-small.en and its model revision. "
    "H1 used greedy decoding (one beam), H2 used "
    "five-beam search, and H3-H5 used stochastic "
    "decoding at temperature 0.3 with recorded seeds "
    "101, 202, and 303. All 2,000 saved hypothesis "
    "metadata rows agreed with the frozen configuration, "
    "and all four batch outputs matched the final "
    "hypothesis dataset without field-level differences."
)

limitations = (
    "The original historical sampling and full-run ASR "
    "inference source code could not be recovered. "
    "Sampling was reconstructed exactly from source "
    "data, whereas ASR reproducibility was verified "
    "at the configuration and saved-artifact levels "
    "without independently replaying inference. "
    "Potential overlap with ASR pretraining data "
    "was not investigated. External validation "
    "was not performed."
)

report = {
    "step": "8.9",
    "status": "PROVENANCE_DOCUMENTED",
    "D07": decisions[0]["status"],
    "D08": decisions[1]["status"],
    "D09": decisions[2]["status"],
    "D10": decisions[3]["status"],
    "D11": decisions[4]["status"],
    "methods_sampling_draft": methods_sampling,
    "methods_asr_draft": methods_asr,
    "limitations_draft": limitations,
    "historical_sampling_code_available": False,
    "historical_asr_code_available": False,
    "independent_asr_replay_performed": False,
    "source_report_sha256": {
        name: sha(path)
        for name, (path, _) in SOURCES.items()
    },
    "models_retrained": False,
    "original_results_modified": False
}

def save_csv_immutable(frame, path):
    tmp = path.with_suffix(".tmp")
    frame.to_csv(tmp, index=False, encoding="utf-8")
    if path.exists():
        assert sha(path) == sha(tmp), (
            f"Existing CSV differs: {path}"
        )
        tmp.unlink()
    else:
        tmp.replace(path)

save_csv_immutable(table, TABLE)
report["decision_table_sha256"] = sha(TABLE)

tmp = RECORD.with_suffix(".tmp")
tmp.write_text(
    json.dumps(report, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

if RECORD.exists():
    assert json.loads(
        RECORD.read_text(encoding="utf-8")
    ) == report
    tmp.unlink()
else:
    tmp.replace(RECORD)

print("=== STEP 8.9 FINAL PROVENANCE RECORD ===")
print("Status:", report["status"])

print("\n=== PROVENANCE DECISIONS ===")
print(
    table[["decision_id", "status"]]
    .to_string(index=False)
)

print("\n=== METHODS: SAMPLING ===")
print(methods_sampling)

print("\n=== METHODS: ASR ===")
print(methods_asr)

print("\n=== LIMITATIONS ===")
print(limitations)

print("\n=== OUTPUT FILES ===")
print("Decision CSV:", TABLE)
print("Record JSON:", RECORD)
print("Record SHA256:", sha(RECORD))
print("Original predictions modified: NO")
print("Models retrained: NO")
